# NIFTY50 Alpha#1: Research to Alpha

This is the single working notebook for the Alpha#1 workflow. It uses real market data from Yahoo Finance via `yfinance`; there is no synthetic data path.

The notebook is designed to answer the question in stages: do we have clean data, does the raw formula point the right way, does the oriented signal have predictive information, do buckets/backtests show useful behavior after costs, does parameter selection survive walk-forward testing, can larger high-volatility universes help, and can the verified but weak signal be rescued through overlays, interactions, and causal gates?

## Workflow Map

1. **Load market data**: current NIFTY50-style symbols plus `^NSEI`, cached locally after download.
2. **Validate data**: missing fields, OHLC consistency, suspicious adjusted returns, usable symbols.
3. **Compute Alpha#1**: the original cross-sectional formula.
4. **Orient the signal**: measure whether raw Alpha#1 is bullish or inverted on forward returns.
5. **Diagnose signal quality**: Rank IC, bucket spread, stability through time.
6. **Backtest simple baskets**: long-only and long/short diagnostics after turnover costs.
7. **Search alpha rules**: thresholds, holding count, rebalance cadence, regime gates.
8. **Walk-forward test**: choose orientation and params on training windows, score only later windows.
9. **Expand the universe**: NIFTY500, dynamic high-volatility baskets, expanded volatile-index parents.
10. **Verify before discard**: formula fixtures, causality audit, corrected metrics, active benchmark attribution.
11. **Run salvage experiments**: smoothed causal overlays, turnover controls, interactions, residualization, and nested regime-gate selection.
12. **Emit candidate/decision tables**: latest names, weights, scores, and decision artifacts.

In [ ]:
from __future__ import annotations

import itertools
import math
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)
pd.options.display.float_format = "{:.6f}".format
plt.style.use("seaborn-v0_8-whitegrid")

try:
    import yfinance as yf
except ImportError as exc:
    raise ImportError("Run `.venv/bin/python -m pip install -r requirements.txt` first.") from exc

START_DATE = "2018-01-01"
END_DATE = None
DEFAULT_HORIZON = 5
FORWARD_HORIZONS = [1, 3, 5, 10]
TOP_N_VALUES = [5, 10]
ANNUALIZATION = 252
COST_BPS_PER_TURNOVER = 10.0
REFRESH_DATA = False

DATA_DIR = Path("research/data/alpha001_nifty50")
ARTIFACT_DIR = Path("research/artifacts/alpha001_research_to_alpha")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

NIFTY50_SYMBOLS = [
    "ADANIENT", "ADANIPORTS", "APOLLOHOSP", "ASIANPAINT", "AXISBANK",
    "BAJAJ-AUTO", "BAJFINANCE", "BAJAJFINSV", "BEL", "BHARTIARTL",
    "CIPLA", "COALINDIA", "DRREDDY", "EICHERMOT", "ETERNAL",
    "GRASIM", "HCLTECH", "HDFCBANK", "HDFCLIFE", "HINDALCO",
    "HINDUNILVR", "ICICIBANK", "INDIGO", "INFY", "ITC",
    "JIOFIN", "JSWSTEEL", "KOTAKBANK", "LT", "M&M",
    "MARUTI", "MAXHEALTH", "NESTLEIND", "NTPC", "ONGC",
    "POWERGRID", "RELIANCE", "SBILIFE", "SHRIRAMFIN", "SBIN",
    "SUNPHARMA", "TATACONSUM", "TATAMOTORS", "TATASTEEL", "TCS",
    "TECHM", "TITAN", "TRENT", "ULTRACEMCO", "WIPRO",
]
YF_TICKERS = [f"{symbol}.NS" for symbol in NIFTY50_SYMBOLS]
SYMBOL_FROM_TICKER = dict(zip(YF_TICKERS, NIFTY50_SYMBOLS))
NIFTY_INDEX_TICKER = "^NSEI"

print(f"Universe symbols configured: {len(NIFTY50_SYMBOLS)}")
print(f"Date range: {START_DATE} to latest available")
print(f"Cost assumption: {COST_BPS_PER_TURNOVER:.1f} bps per unit of turnover")

## 1. Load Real Market Data

This cell downloads adjusted and raw OHLCV data from Yahoo Finance if local CSV caches are absent or `REFRESH_DATA=True`. Once cached, later runs reuse the local files so you can iterate quickly.

In [ ]:
FIELD_FILES = {
    "open": DATA_DIR / "open.csv",
    "high": DATA_DIR / "high.csv",
    "low": DATA_DIR / "low.csv",
    "close": DATA_DIR / "close.csv",
    "adj_close": DATA_DIR / "adj_close.csv",
    "volume": DATA_DIR / "volume.csv",
    "nifty_close": DATA_DIR / "nifty_close.csv",
}


def read_frame(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, index_col=0, parse_dates=True)
    frame.index = pd.to_datetime(frame.index).tz_localize(None)
    return frame.sort_index()


def write_frame(frame: pd.DataFrame | pd.Series, path: Path) -> None:
    out = frame.to_frame() if isinstance(frame, pd.Series) else frame
    out.to_csv(path)


def extract_series(raw_data: pd.DataFrame, ticker: str, field: str) -> pd.Series | None:
    if raw_data.empty:
        return None
    if isinstance(raw_data.columns, pd.MultiIndex):
        candidates = [(ticker, field), (field, ticker)]
        for key in candidates:
            if key in raw_data.columns:
                return raw_data[key].astype(float)
        return None
    if field in raw_data.columns:
        return raw_data[field].astype(float)
    return None


def extract_field(raw_data: pd.DataFrame, field: str) -> pd.DataFrame:
    frames = {}
    for ticker in YF_TICKERS:
        series = extract_series(raw_data, ticker, field)
        if series is not None:
            frames[SYMBOL_FROM_TICKER[ticker]] = series
    frame = pd.DataFrame(frames).sort_index()
    frame.index = pd.to_datetime(frame.index).tz_localize(None)
    return frame.loc[~frame.index.duplicated(keep="last")]


def download_and_cache() -> dict[str, pd.DataFrame | pd.Series]:
    tickers = YF_TICKERS + [NIFTY_INDEX_TICKER]
    raw = yf.download(
        tickers=tickers,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=False,
        group_by="ticker",
        threads=True,
        progress=False,
    )
    if raw.empty:
        raise ValueError("No data downloaded from yfinance. Check network access and ticker availability.")

    data = {
        "open": extract_field(raw, "Open"),
        "high": extract_field(raw, "High"),
        "low": extract_field(raw, "Low"),
        "close": extract_field(raw, "Close"),
        "adj_close": extract_field(raw, "Adj Close"),
        "volume": extract_field(raw, "Volume"),
    }
    nifty_close = extract_series(raw, NIFTY_INDEX_TICKER, "Close")
    if nifty_close is None:
        nifty_raw = yf.download(NIFTY_INDEX_TICKER, start=START_DATE, end=END_DATE, auto_adjust=False, progress=False)
        nifty_close = extract_series(nifty_raw, NIFTY_INDEX_TICKER, "Close") or extract_series(nifty_raw, "Close", "Close")
    if nifty_close is None:
        raise ValueError("Could not extract NIFTY index close from yfinance output.")
    nifty_close.index = pd.to_datetime(nifty_close.index).tz_localize(None)
    data["nifty_close"] = nifty_close.rename("nifty_close").sort_index()

    for name, frame in data.items():
        write_frame(frame, FIELD_FILES[name])
    return data


def load_market_data() -> dict[str, pd.DataFrame | pd.Series]:
    cache_ready = all(path.exists() for path in FIELD_FILES.values())
    if cache_ready and not REFRESH_DATA:
        data = {name: read_frame(path) for name, path in FIELD_FILES.items()}
        data["nifty_close"] = data["nifty_close"].iloc[:, 0].rename("nifty_close")
        print("Loaded cached market data from", DATA_DIR)
        return data
    print("Downloading real market data with yfinance...")
    return download_and_cache()

market = load_market_data()
open_px_raw = market["open"]
high_px_raw = market["high"]
low_px_raw = market["low"]
close_px_raw = market["close"]
adj_close_px = market["adj_close"]
volume = market["volume"]
nifty_close = market["nifty_close"]

print("Downloaded/cached field shapes:")
display(pd.Series({
    "open": open_px_raw.shape,
    "high": high_px_raw.shape,
    "low": low_px_raw.shape,
    "close": close_px_raw.shape,
    "adj_close": adj_close_px.shape,
    "volume": volume.shape,
    "nifty_close": nifty_close.shape,
}).to_frame("shape"))
display(adj_close_px.tail())

## 2. Data Validation Checkpoint

Before using the alpha, we check whether the downloaded panel is usable. This is where we identify missing symbols, malformed OHLC rows, non-positive prices, suspicious adjusted returns, and symbols with too little history.

In [ ]:
def validate_ohlcv_panel(open_px, high_px, low_px, close_px, adj_close, volume_df) -> dict[str, pd.DataFrame]:
    all_symbols = sorted(set(open_px.columns) | set(high_px.columns) | set(low_px.columns) | set(close_px.columns) | set(adj_close.columns) | set(volume_df.columns))
    reports = {}
    missing = pd.DataFrame({
        "open_missing": open_px.reindex(columns=all_symbols).isna().mean(),
        "high_missing": high_px.reindex(columns=all_symbols).isna().mean(),
        "low_missing": low_px.reindex(columns=all_symbols).isna().mean(),
        "close_missing": close_px.reindex(columns=all_symbols).isna().mean(),
        "adj_close_missing": adj_close.reindex(columns=all_symbols).isna().mean(),
        "volume_missing": volume_df.reindex(columns=all_symbols).isna().mean(),
    }).sort_values("adj_close_missing", ascending=False)
    reports["missing_ratio"] = missing

    ohlc_rows = []
    for symbol in all_symbols:
        if symbol not in open_px or symbol not in high_px or symbol not in low_px or symbol not in close_px:
            continue
        frame = pd.concat({
            "open": open_px[symbol], "high": high_px[symbol], "low": low_px[symbol], "close": close_px[symbol]
        }, axis=1).dropna()
        bad = frame[(frame["high"] < frame[["open", "close"]].max(axis=1)) | (frame["low"] > frame[["open", "close"]].min(axis=1)) | (frame["low"] > frame["high"])]
        for date, row in bad.head(10).iterrows():
            ohlc_rows.append({"date": date, "symbol": symbol, **row.to_dict()})
    reports["ohlc_inconsistencies"] = pd.DataFrame(ohlc_rows)

    nonpositive_rows = []
    for label, frame in {"open": open_px, "high": high_px, "low": low_px, "close": close_px, "adj_close": adj_close}.items():
        bad = frame.where(frame <= 0).stack().dropna().rename("value").reset_index()
        if not bad.empty:
            bad.columns = ["date", "symbol", "value"]
            bad["field"] = label
            nonpositive_rows.append(bad)
    reports["nonpositive_prices"] = pd.concat(nonpositive_rows, ignore_index=True) if nonpositive_rows else pd.DataFrame(columns=["date", "symbol", "value", "field"])

    suspicious_rows = []
    adj_returns = adj_close.pct_change(fill_method=None)
    for symbol in adj_returns.columns:
        bad = adj_returns[symbol].abs().gt(0.30)
        for date, value in adj_returns.loc[bad, symbol].head(10).items():
            suspicious_rows.append({"date": date, "symbol": symbol, "adjusted_return": value})
    reports["suspicious_adjusted_returns"] = pd.DataFrame(suspicious_rows)
    return reports


dq_reports = validate_ohlcv_panel(open_px_raw, high_px_raw, low_px_raw, close_px_raw, adj_close_px, volume)

summary = pd.Series({
    "configured_symbols": len(NIFTY50_SYMBOLS),
    "symbols_with_adjusted_close": adj_close_px.shape[1],
    "raw_calendar_days": len(adj_close_px),
    "first_date": adj_close_px.index.min(),
    "last_date": adj_close_px.index.max(),
    "ohlc_inconsistency_rows_shown": len(dq_reports["ohlc_inconsistencies"]),
    "nonpositive_price_rows": len(dq_reports["nonpositive_prices"]),
    "suspicious_adjusted_return_rows_shown": len(dq_reports["suspicious_adjusted_returns"]),
})
display(summary.to_frame("data_checkpoint"))

display(dq_reports["missing_ratio"].head(15))
if not dq_reports["ohlc_inconsistencies"].empty:
    display(dq_reports["ohlc_inconsistencies"].head(20))
if not dq_reports["suspicious_adjusted_returns"].empty:
    display(dq_reports["suspicious_adjusted_returns"].head(20))

MAX_MISSING_RATIO = 0.45
bad_symbols = set(dq_reports["missing_ratio"].query("adj_close_missing > @MAX_MISSING_RATIO").index)
bad_symbols |= set(dq_reports["nonpositive_prices"]["symbol"].unique())
usable_symbols = [symbol for symbol in NIFTY50_SYMBOLS if symbol in adj_close_px.columns and symbol not in bad_symbols]

return_price = adj_close_px[usable_symbols].copy()
alpha_close = return_price.copy()
raw_close = close_px_raw.reindex(columns=usable_symbols)
volume = volume.reindex(columns=usable_symbols)

print(f"Usable symbols after validation: {len(usable_symbols)} / {len(NIFTY50_SYMBOLS)}")
print("Removed symbols:", sorted(set(NIFTY50_SYMBOLS) - set(usable_symbols)))
print(f"Research date range: {return_price.index.min().date()} to {return_price.index.max().date()}")

## 3. Compute Raw Alpha#1

Alpha#1 is a cross-sectional formula. Each date ranks the universe by the recent position of a transformed price/volatility input. At this stage we do **not** assume that high raw score is bullish.

In [ ]:
def signed_power(frame: pd.DataFrame, power: float) -> pd.DataFrame:
    return np.sign(frame) * (frame.abs() ** power)


def ts_argmax_recent_position(frame: pd.DataFrame, window: int) -> pd.DataFrame:
    def position(values: np.ndarray) -> float:
        if np.isnan(values).all():
            return np.nan
        return float(np.nanargmax(values) + 1)
    return frame.rolling(window, min_periods=window).apply(position, raw=True)


def rank_cross_sectional(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.rank(axis=1, pct=True)


def compute_alpha001(close: pd.DataFrame, returns: pd.DataFrame) -> pd.DataFrame:
    vol20 = returns.rolling(20, min_periods=20).std()
    condition_value = close.where(~returns.lt(0), vol20)
    transformed = signed_power(condition_value, 2)
    argmax5 = ts_argmax_recent_position(transformed, 5)
    return rank_cross_sectional(argmax5) - 0.5


def forward_return(price: pd.DataFrame, horizon: int) -> pd.DataFrame:
    return price.shift(-horizon).div(price).sub(1.0)

returns = return_price.pct_change(fill_method=None)
raw_alpha001 = compute_alpha001(alpha_close, returns)
forward_returns = {h: forward_return(return_price, h) for h in FORWARD_HORIZONS}

raw_alpha001.to_csv(ARTIFACT_DIR / "raw_alpha001_scores.csv")
print("Raw Alpha#1 score distribution:")
display(raw_alpha001.stack().describe().to_frame("raw_alpha001"))
display(raw_alpha001.tail())

## 4. Measure Direction and Orient the Signal

This is the first important research decision. We compute daily cross-sectional Rank IC between raw Alpha#1 and future returns. If the raw 5-day IC is negative, the tradable signal is inverted so that higher oriented score always means “more bullish.”

In [ ]:
def cross_sectional_corr_by_date(signal: pd.DataFrame, future: pd.DataFrame, method: str = "spearman", min_names: int = 10) -> pd.Series:
    rows = []
    for date in signal.index.intersection(future.index):
        s = signal.loc[date]
        f = future.loc[date]
        valid = s.notna() & f.notna()
        if valid.sum() < min_names:
            rows.append((date, np.nan))
            continue
        x = s[valid].astype(float)
        y = f[valid].astype(float)
        if method == "spearman":
            x = x.rank()
            y = y.rank()
        rows.append((date, x.corr(y, method="pearson")))
    return pd.Series(dict(rows)).sort_index().rename("rank_ic")


def ic_summary(signal: pd.DataFrame, fwd_by_horizon: dict[int, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    series_by_horizon = {}
    for horizon, fwd in fwd_by_horizon.items():
        rank_ic = cross_sectional_corr_by_date(signal, fwd, method="spearman")
        pearson_ic = cross_sectional_corr_by_date(signal, fwd, method="pearson")
        series_by_horizon[horizon] = rank_ic
        rows.append({
            "horizon_days": horizon,
            "mean_rank_ic": rank_ic.mean(),
            "median_rank_ic": rank_ic.median(),
            "rank_ic_vol": rank_ic.std(),
            "rank_icir": rank_ic.mean() / rank_ic.std() if rank_ic.std() and not pd.isna(rank_ic.std()) else np.nan,
            "positive_rank_ic_rate": rank_ic.dropna().gt(0).mean(),
            "mean_pearson_ic": pearson_ic.mean(),
            "observations": rank_ic.notna().sum(),
        })
    return pd.DataFrame(rows).set_index("horizon_days"), series_by_horizon

raw_ic_summary, raw_rank_ic_by_horizon = ic_summary(raw_alpha001, forward_returns)
default_raw_ic = raw_ic_summary.loc[DEFAULT_HORIZON, "mean_rank_ic"]
SIGNAL_DIRECTION = 1 if default_raw_ic >= 0 else -1
orientation_label = "raw" if SIGNAL_DIRECTION == 1 else "inverted"
oriented_alpha001 = SIGNAL_DIRECTION * raw_alpha001
oriented_ic_summary, oriented_rank_ic_by_horizon = ic_summary(oriented_alpha001, forward_returns)

raw_ic_summary.to_csv(ARTIFACT_DIR / "raw_ic_summary.csv")
oriented_ic_summary.to_csv(ARTIFACT_DIR / "oriented_ic_summary.csv")
oriented_alpha001.to_csv(ARTIFACT_DIR / "oriented_alpha001_scores.csv")

print(f"Default horizon: {DEFAULT_HORIZON} trading days")
print(f"Raw mean Rank IC at default horizon: {default_raw_ic:.6f}")
print(f"Selected tradable orientation: {orientation_label}")
print("Interpretation after orientation: higher score = more bullish")
print("Raw IC summary")
display(raw_ic_summary)
print("Oriented IC summary")
display(oriented_ic_summary)

fig, ax = plt.subplots(figsize=(12, 5))
for horizon, series in oriented_rank_ic_by_horizon.items():
    series.rolling(63).mean().plot(ax=ax, label=f"{horizon}d")
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set_title("Rolling 63-day Rank IC after orientation")
ax.set_ylabel("Rolling mean Rank IC")
ax.legend()
plt.show()

## 5. Bucket Diagnostics

Buckets answer a plain question: after orientation, do the top-ranked names beat the bottom-ranked names? Multi-day bucket returns overlap, so this is a signal diagnostic rather than an executable portfolio P&L.

In [ ]:
def bucket_returns(signal: pd.DataFrame, future: pd.DataFrame, top_n: int) -> pd.DataFrame:
    rows = []
    for date in signal.index.intersection(future.index):
        scores = signal.loc[date].dropna()
        rets = future.loc[date].dropna()
        common = scores.index.intersection(rets.index)
        if len(common) < top_n * 2:
            continue
        scores = scores[common].sort_values(ascending=False)
        rets = rets[common]
        top_symbols = scores.head(top_n).index
        bottom_symbols = scores.tail(top_n).index
        rows.append({
            "date": date,
            "top_return": rets[top_symbols].mean(),
            "bottom_return": rets[bottom_symbols].mean(),
            "spread_return": rets[top_symbols].mean() - rets[bottom_symbols].mean(),
            "top_symbols": ",".join(top_symbols),
            "bottom_symbols": ",".join(bottom_symbols),
        })
    columns = ["date", "top_return", "bottom_return", "spread_return", "top_symbols", "bottom_symbols"]
    return pd.DataFrame(rows, columns=columns).set_index("date")

bucket_frames = {}
bucket_rows = []
for horizon, fwd in forward_returns.items():
    for top_n in TOP_N_VALUES:
        frame = bucket_returns(oriented_alpha001, fwd, top_n)
        bucket_frames[(horizon, top_n)] = frame
        bucket_rows.append({
            "horizon_days": horizon,
            "top_n": top_n,
            "mean_top_return": frame["top_return"].mean(),
            "mean_bottom_return": frame["bottom_return"].mean(),
            "mean_spread_return": frame["spread_return"].mean(),
            "spread_hit_rate": frame["spread_return"].gt(0).mean(),
            "observations": len(frame),
        })

bucket_summary = pd.DataFrame(bucket_rows)
bucket_summary.to_csv(ARTIFACT_DIR / "bucket_summary.csv", index=False)
display(bucket_summary)

fig, ax = plt.subplots(figsize=(12, 5))
for horizon in FORWARD_HORIZONS:
    frame = bucket_frames[(horizon, 10)]
    if not frame.empty:
        frame["spread_return"].fillna(0).cumsum().plot(ax=ax, label=f"{horizon}d top-bottom")
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set_title("Cumulative diagnostic bucket spread: oriented Alpha#1, Top 10 minus Bottom 10")
ax.set_ylabel("Cumulative spread approximation")
ax.legend()
plt.show()

## 6. Simple Basket Backtests

This converts the oriented signal into portfolios. The convention here is: compute the signal using data through date `t`, then measure the next session return aligned to `t`. Costs are charged from turnover. Treat this as a daily research simulator; a production backtest should replace it with explicit open/close execution data.

In [ ]:
next_session_returns = return_price.pct_change(fill_method=None).shift(-1)


def build_weights(signal: pd.DataFrame, top_n: int, long_short: bool, min_signal: float = -np.inf) -> pd.DataFrame:
    weights = pd.DataFrame(0.0, index=signal.index, columns=signal.columns)
    for date in signal.index:
        scores = signal.loc[date].dropna().sort_values(ascending=False)
        scores = scores[scores >= min_signal]
        required = top_n * (2 if long_short else 1)
        if len(scores) < required:
            continue
        top = scores.head(top_n).index
        weights.loc[date, top] = 1.0 / top_n
        if long_short:
            bottom = scores.tail(top_n).index
            weights.loc[date, bottom] = -1.0 / top_n
    return weights


def performance_metrics(returns: pd.Series, turnover: pd.Series | None = None, periods_per_year: int = 252) -> dict:
    r = returns.dropna()
    if r.empty:
        return {}
    equity = (1.0 + r).cumprod()
    years = len(r) / periods_per_year
    total_return = equity.iloc[-1] - 1.0
    cagr = equity.iloc[-1] ** (1.0 / years) - 1.0 if years > 0 and equity.iloc[-1] > 0 else np.nan
    annual_vol = r.std(ddof=0) * math.sqrt(periods_per_year)
    sharpe = (r.mean() * periods_per_year) / annual_vol if annual_vol and annual_vol > 0 else np.nan
    downside = r[r < 0].std(ddof=0) * math.sqrt(periods_per_year)
    sortino = (r.mean() * periods_per_year) / downside if downside and downside > 0 else np.nan
    drawdown = equity / equity.cummax() - 1.0
    return {
        "observations": len(r),
        "total_return": total_return,
        "cagr": cagr,
        "annual_vol": annual_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": drawdown.min(),
        "daily_win_rate": r.gt(0).mean(),
        "mean_daily_return": r.mean(),
        "avg_daily_turnover": turnover.reindex(r.index).mean() if turnover is not None else np.nan,
    }


def backtest_weights(weights: pd.DataFrame, next_returns: pd.DataFrame, cost_bps: float = COST_BPS_PER_TURNOVER) -> dict:
    aligned_returns = next_returns.reindex_like(weights)
    valid_exposure_return = weights.abs().gt(0) & aligned_returns.notna()
    no_exposure = weights.abs().sum(axis=1).eq(0)
    gross = weights.mul(aligned_returns).sum(axis=1, min_count=1).where(valid_exposure_return.any(axis=1) | no_exposure)
    turnover = weights.diff().abs().sum(axis=1, min_count=1).fillna(weights.abs().sum(axis=1))
    costs = turnover * (cost_bps / 10000.0)
    net = gross - costs
    valid_index = net.dropna().index
    return {
        "returns": net.reindex(valid_index),
        "gross_returns": gross.reindex(valid_index),
        "turnover": turnover.reindex(valid_index).fillna(0),
        "costs": costs.reindex(valid_index).fillna(0),
        "weights": weights,
        "metrics": performance_metrics(net, turnover),
    }

strategy_returns = {}
strategy_turnover = {}
metric_rows = []
for top_n in TOP_N_VALUES:
    for long_short in [False, True]:
        name = f"{'long_short' if long_short else 'long_only'}_top{top_n}"
        weights = build_weights(oriented_alpha001, top_n=top_n, long_short=long_short)
        bt = backtest_weights(weights, next_session_returns)
        strategy_returns[name] = bt["returns"]
        strategy_turnover[name] = bt["turnover"]
        metric_rows.append({
            "strategy": name,
            "top_n": top_n,
            "long_short": long_short,
            "signal_orientation": orientation_label,
            "cost_bps_per_turnover": COST_BPS_PER_TURNOVER,
            **bt["metrics"],
        })

strategy_metrics = pd.DataFrame(metric_rows).set_index("strategy")
strategy_metrics.to_csv(ARTIFACT_DIR / "simple_strategy_metrics.csv")
display(strategy_metrics)

fig, ax = plt.subplots(figsize=(12, 6))
for name, rets in strategy_returns.items():
    (1 + rets.fillna(0)).cumprod().plot(ax=ax, label=name)
ax.set_title("Simple Alpha#1 basket equity curves after orientation and costs")
ax.set_ylabel("Growth of 1")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
for name, rets in strategy_returns.items():
    equity = (1 + rets.fillna(0)).cumprod()
    (equity / equity.cummax() - 1).plot(ax=ax, label=name)
ax.set_title("Simple Alpha#1 basket drawdowns")
ax.set_ylabel("Drawdown")
ax.legend()
plt.show()

## 7. Regimes and Rule Search

Now we ask whether the signal works better with thresholding, fewer holdings, slower rebalancing, or regime gates. The full-sample leaderboard is an exploration tool; the walk-forward section below is the more important test.

In [ ]:
def make_regime_frame(index_close: pd.Series) -> pd.DataFrame:
    n = index_close.dropna().copy()
    sma50 = n.rolling(50, min_periods=50).mean()
    sma200 = n.rolling(200, min_periods=200).mean()
    index_ret = n.pct_change(fill_method=None)
    vol20 = index_ret.rolling(20, min_periods=20).std() * math.sqrt(252)
    vol_history = vol20.shift(1)
    vol_q_low = vol_history.rolling(252, min_periods=100).quantile(0.33)
    vol_q_high = vol_history.rolling(252, min_periods=100).quantile(0.67)

    trend = pd.Series("sideways", index=n.index, dtype="object")
    trend[(n > sma50) & (sma50 > sma200)] = "bull"
    trend[(n < sma50) & (sma50 < sma200)] = "bear"
    trend[sma50.isna() | sma200.isna()] = "unknown"

    vol_regime = pd.Series("medium_vol", index=n.index, dtype="object")
    vol_regime[vol20 <= vol_q_low] = "low_vol"
    vol_regime[vol20 >= vol_q_high] = "high_vol"
    vol_regime[vol20.isna() | vol_q_low.isna() | vol_q_high.isna()] = "unknown"

    return pd.DataFrame({
        "nifty_close": n,
        "nifty_ret": index_ret,
        "nifty_sma50": sma50,
        "nifty_sma200": sma200,
        "nifty_vol20_ann": vol20,
        "trend_regime": trend,
        "vol_regime": vol_regime,
    })


def allowed_by_regime(row: pd.Series, regime_filter: str) -> bool:
    if regime_filter == "all":
        return True
    if regime_filter == "nifty_bull_or_sideways":
        return row.get("trend_regime") in {"bull", "sideways"}
    if regime_filter == "nifty_bull_only":
        return row.get("trend_regime") == "bull"
    if regime_filter == "not_high_vol":
        return row.get("vol_regime") != "high_vol"
    if regime_filter == "bull_and_not_high_vol":
        return row.get("trend_regime") == "bull" and row.get("vol_regime") != "high_vol"
    raise ValueError(f"Unknown regime_filter: {regime_filter}")

@dataclass(frozen=True)
class StrategyParams:
    top_n: int
    min_signal: float
    rebalance_every: int
    regime_filter: str
    long_short: bool = False
    cost_bps: float = COST_BPS_PER_TURNOVER


def build_rule_weights(signal: pd.DataFrame, regimes: pd.DataFrame, params: StrategyParams) -> pd.DataFrame:
    weights = pd.DataFrame(0.0, index=signal.index, columns=signal.columns)
    last_weights = pd.Series(0.0, index=signal.columns)
    for i, date in enumerate(signal.index):
        if i % params.rebalance_every != 0:
            weights.loc[date] = last_weights
            continue
        if date not in regimes.index or not allowed_by_regime(regimes.loc[date], params.regime_filter):
            last_weights = pd.Series(0.0, index=signal.columns)
            weights.loc[date] = last_weights
            continue
        scores = signal.loc[date].dropna().sort_values(ascending=False)
        scores = scores[scores >= params.min_signal]
        required = params.top_n * (2 if params.long_short else 1)
        if len(scores) < required:
            last_weights = pd.Series(0.0, index=signal.columns)
            weights.loc[date] = last_weights
            continue
        new_weights = pd.Series(0.0, index=signal.columns)
        top = scores.head(params.top_n).index
        new_weights.loc[top] = 1.0 / params.top_n
        if params.long_short:
            bottom = scores.tail(params.top_n).index
            new_weights.loc[bottom] = -1.0 / params.top_n
        last_weights = new_weights
        weights.loc[date] = last_weights
    return weights


def backtest_rule(signal: pd.DataFrame, next_returns: pd.DataFrame, regimes: pd.DataFrame, params: StrategyParams) -> dict:
    common = signal.index.intersection(next_returns.index).intersection(regimes.index)
    weights = build_rule_weights(signal.loc[common], regimes.loc[common], params)
    return backtest_weights(weights, next_returns.loc[common], cost_bps=params.cost_bps)


def rule_grid() -> list[StrategyParams]:
    rows = []
    for top_n, min_signal, rebalance_every, regime_filter, long_short in itertools.product(
        [5, 10],
        [0.00, 0.10, 0.20, 0.30],
        [1, 3, 5],
        ["all", "nifty_bull_or_sideways", "not_high_vol"],
        [False, True],
    ):
        rows.append(StrategyParams(top_n=top_n, min_signal=min_signal, rebalance_every=rebalance_every, regime_filter=regime_filter, long_short=long_short))
    return rows


def score_metrics(metrics: dict) -> float:
    sharpe = metrics.get("sharpe", np.nan)
    max_dd = abs(metrics.get("max_drawdown", np.nan))
    turnover = metrics.get("avg_daily_turnover", np.nan)
    total_return = metrics.get("total_return", np.nan)
    return (
        pd.Series([sharpe]).fillna(-999).iloc[0]
        + 0.20 * pd.Series([total_return]).replace([np.inf, -np.inf], np.nan).fillna(-999).iloc[0]
        - 1.50 * pd.Series([max_dd]).fillna(1).iloc[0]
        - 0.10 * pd.Series([turnover]).fillna(1).iloc[0]
    )


def evaluate_grid(signal: pd.DataFrame, next_returns: pd.DataFrame, regimes: pd.DataFrame, grid: list[StrategyParams]) -> pd.DataFrame:
    rows = []
    for params in grid:
        bt = backtest_rule(signal, next_returns, regimes, params)
        metrics = bt["metrics"]
        exposure = bt["weights"].abs().sum(axis=1).gt(0).mean()
        rows.append({
            "top_n": params.top_n,
            "min_signal": params.min_signal,
            "rebalance_every": params.rebalance_every,
            "regime_filter": params.regime_filter,
            "long_short": params.long_short,
            "exposure": exposure,
            **metrics,
            "score": score_metrics(metrics),
        })
    return pd.DataFrame(rows)


def params_from_row(row: pd.Series) -> StrategyParams:
    return StrategyParams(
        top_n=int(row["top_n"]),
        min_signal=float(row["min_signal"]),
        rebalance_every=int(row["rebalance_every"]),
        regime_filter=str(row["regime_filter"]),
        long_short=bool(row["long_short"]),
    )

regimes = make_regime_frame(nifty_close)
display(regimes.tail())
display(regimes[["trend_regime", "vol_regime"]].value_counts().to_frame("days"))

GRID = rule_grid()
search_results = evaluate_grid(oriented_alpha001, next_session_returns, regimes, GRID)
leaderboard = search_results.query("observations >= 252 and exposure >= 0.05").sort_values(["score", "sharpe"], ascending=[False, False]).head(25)
search_results.to_csv(ARTIFACT_DIR / "rule_search_results.csv", index=False)
leaderboard.to_csv(ARTIFACT_DIR / "rule_search_leaderboard.csv", index=False)
display(leaderboard)

best_in_sample_params = params_from_row(leaderboard.iloc[0])
best_in_sample_bt = backtest_rule(oriented_alpha001, next_session_returns, regimes, best_in_sample_params)
print("Best full-sample research params:", best_in_sample_params)
display(pd.Series(best_in_sample_bt["metrics"]).to_frame("best_full_sample_research"))

## 8. Walk-Forward Validation

This is the most important performance checkpoint. Each fold chooses signal direction and strategy parameters using only the training window, then scores the selected rule on the later test window.

In [ ]:
def rolling_windows(index: pd.Index, train_days: int = 504, test_days: int = 126, step_days: int = 126):
    idx = pd.Index(index).sort_values()
    start = 0
    while start + train_days + test_days <= len(idx):
        train_idx = idx[start:start + train_days]
        test_idx = idx[start + train_days:start + train_days + test_days]
        yield train_idx, test_idx
        start += step_days


def choose_direction_on_training(raw_signal: pd.DataFrame, train_future: pd.DataFrame) -> tuple[int, float]:
    train_ic = cross_sectional_corr_by_date(raw_signal, train_future, method="spearman")
    mean_ic = train_ic.mean()
    direction = 1 if mean_ic >= 0 else -1
    return direction, mean_ic


def walk_forward_optimize(raw_signal: pd.DataFrame, next_returns: pd.DataFrame, regimes: pd.DataFrame, grid: list[StrategyParams]) -> tuple[pd.DataFrame, pd.Series]:
    common = raw_signal.index.intersection(next_returns.index).intersection(regimes.index)
    fwd_for_orientation = forward_returns[DEFAULT_HORIZON].reindex(index=raw_signal.index, columns=raw_signal.columns)
    rows = []
    oos_returns = []
    for fold, (train_idx, test_idx) in enumerate(rolling_windows(common), start=1):
        direction, train_raw_ic = choose_direction_on_training(raw_signal.loc[train_idx], fwd_for_orientation.loc[train_idx])
        train_signal = direction * raw_signal.loc[train_idx]
        test_signal = direction * raw_signal.loc[test_idx]

        train_results = evaluate_grid(train_signal, next_returns.loc[train_idx], regimes.loc[train_idx], grid)
        train_results = train_results.query("observations >= 126 and exposure >= 0.05")
        if train_results.empty:
            continue
        chosen = train_results.sort_values(["score", "sharpe"], ascending=[False, False]).iloc[0]
        params = params_from_row(chosen)
        test_bt = backtest_rule(test_signal, next_returns.loc[test_idx], regimes.loc[test_idx], params)
        test_returns = test_bt["returns"]
        oos_returns.append(test_returns)
        rows.append({
            "fold": fold,
            "train_start": train_idx[0],
            "train_end": train_idx[-1],
            "test_start": test_idx[0],
            "test_end": test_idx[-1],
            "train_raw_mean_ic": train_raw_ic,
            "chosen_orientation": "raw" if direction == 1 else "inverted",
            "chosen_top_n": params.top_n,
            "chosen_min_signal": params.min_signal,
            "chosen_rebalance_every": params.rebalance_every,
            "chosen_regime_filter": params.regime_filter,
            "chosen_long_short": params.long_short,
            **{f"test_{k}": v for k, v in test_bt["metrics"].items()},
        })
    combined = pd.concat(oos_returns).sort_index() if oos_returns else pd.Series(dtype=float, name="walk_forward_oos_return")
    combined = combined[~combined.index.duplicated(keep="last")].rename("walk_forward_oos_return")
    return pd.DataFrame(rows), combined

WF_GRID = [params for params in GRID if params.top_n in {5, 10} and params.rebalance_every in {1, 3, 5}]
wf_report, wf_returns = walk_forward_optimize(raw_alpha001, next_session_returns, regimes, WF_GRID)
wf_metrics = performance_metrics(wf_returns, pd.Series(index=wf_returns.index, data=np.nan)) if not wf_returns.empty else {}

wf_report.to_csv(ARTIFACT_DIR / "walk_forward_report.csv", index=False)
wf_returns.to_csv(ARTIFACT_DIR / "walk_forward_oos_returns.csv")

display(wf_report)
print("Walk-forward out-of-sample metrics")
display(pd.Series(wf_metrics).to_frame("walk_forward_oos"))

if not wf_returns.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    (1 + wf_returns.fillna(0)).cumprod().plot(ax=ax, color="#174A7C")
    ax.set_title("Walk-forward out-of-sample equity curve")
    ax.set_ylabel("Growth of 1")
    plt.show()

    fig, ax = plt.subplots(figsize=(12, 4))
    equity = (1 + wf_returns.fillna(0)).cumprod()
    (equity / equity.cummax() - 1).plot(ax=ax, color="#C73E1D")
    ax.set_title("Walk-forward out-of-sample drawdown")
    ax.set_ylabel("Drawdown")
    plt.show()

if not wf_report.empty:
    print("Parameter stability across folds")
    for col in ["chosen_orientation", "chosen_top_n", "chosen_min_signal", "chosen_rebalance_every", "chosen_regime_filter", "chosen_long_short"]:
        display(wf_report[col].value_counts(dropna=False).to_frame("fold_count"))

## 9. Final Alpha Candidate Table

For the latest signal date, we choose parameters using the most recent training window and produce a candidate table. This is still a research output: it tells us what Alpha#1 would currently prefer under the learned rule, not an instruction to trade without portfolio and risk review.

In [ ]:
def stock_level_alpha_report(signal: pd.DataFrame, future_return: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    rows = []
    top_mask = pd.DataFrame(False, index=signal.index, columns=signal.columns)
    bottom_mask = pd.DataFrame(False, index=signal.index, columns=signal.columns)
    for date in signal.index:
        scores = signal.loc[date].dropna().sort_values(ascending=False)
        if len(scores) < top_n * 2:
            continue
        top_mask.loc[date, scores.head(top_n).index] = True
        bottom_mask.loc[date, scores.tail(top_n).index] = True
    for symbol in signal.columns:
        pair = pd.concat([signal[symbol].rename("oriented_alpha"), future_return[symbol].rename("fwd")], axis=1).dropna()
        top_rets = future_return[symbol][top_mask[symbol]].dropna()
        bottom_rets = future_return[symbol][bottom_mask[symbol]].dropna()
        rows.append({
            "symbol": symbol,
            "alpha_fwd_corr": pair["oriented_alpha"].rank().corr(pair["fwd"].rank(), method="pearson") if len(pair) >= 30 else np.nan,
            "top_selected_mean_fwd_return": top_rets.mean(),
            "top_selected_hit_rate": top_rets.gt(0).mean() if len(top_rets) else np.nan,
            "top_selected_count": len(top_rets),
            "bottom_selected_mean_fwd_return": bottom_rets.mean(),
            "bottom_selected_hit_rate": bottom_rets.gt(0).mean() if len(bottom_rets) else np.nan,
            "bottom_selected_count": len(bottom_rets),
        })
    report = pd.DataFrame(rows).set_index("symbol")
    components = pd.DataFrame(index=report.index)
    components["corr"] = report["alpha_fwd_corr"].rank(pct=True)
    components["top_return"] = report["top_selected_mean_fwd_return"].rank(pct=True)
    components["top_hit"] = report["top_selected_hit_rate"].rank(pct=True)
    components["avoid_bottom"] = (-report["bottom_selected_mean_fwd_return"]).rank(pct=True)
    components["sample"] = report["top_selected_count"].rank(pct=True)
    report["alpha001_stock_score"] = (
        0.30 * components["corr"]
        + 0.30 * components["top_return"]
        + 0.20 * components["top_hit"]
        + 0.10 * components["avoid_bottom"]
        + 0.10 * components["sample"]
    )
    return report.sort_values("alpha001_stock_score", ascending=False)


def minmax_score(series: pd.Series) -> pd.Series:
    series = series.astype(float)
    lo, hi = series.min(), series.max()
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(0.5, index=series.index)
    return (series - lo) / (hi - lo)

latest_date = raw_alpha001.dropna(how="all").index[-1]
common = raw_alpha001.index.intersection(next_session_returns.index).intersection(regimes.index)
recent_train_idx = common[common <= latest_date][-504:]
final_direction, final_train_ic = choose_direction_on_training(raw_alpha001.loc[recent_train_idx], forward_returns[DEFAULT_HORIZON].loc[recent_train_idx])
final_signal = final_direction * raw_alpha001
final_orientation_label = "raw" if final_direction == 1 else "inverted"

final_train_results = evaluate_grid(final_signal.loc[recent_train_idx], next_session_returns.loc[recent_train_idx], regimes.loc[recent_train_idx], WF_GRID)
final_train_results = final_train_results.query("observations >= 126 and exposure >= 0.05").sort_values(["score", "sharpe"], ascending=[False, False])
final_params = params_from_row(final_train_results.iloc[0])

stock_ranking = stock_level_alpha_report(final_signal, forward_returns[DEFAULT_HORIZON], top_n=10)
stock_score_scaled = minmax_score(stock_ranking["alpha001_stock_score"])

latest_scores = final_signal.loc[latest_date].dropna().sort_values(ascending=False)
latest_regime = regimes.reindex(final_signal.index).loc[latest_date]
regime_allowed = allowed_by_regime(latest_regime, final_params.regime_filter)
selected_scores = latest_scores[latest_scores >= final_params.min_signal].head(final_params.top_n) if regime_allowed else pd.Series(dtype=float)

recent_ic_series = cross_sectional_corr_by_date(final_signal.loc[recent_train_idx], forward_returns[DEFAULT_HORIZON].loc[recent_train_idx], method="spearman")
recent_ic_score = float(np.clip((recent_ic_series.tail(63).mean() + 0.05) / 0.10, 0, 1)) if len(recent_ic_series) else 0.5

candidate_rows = []
for symbol, score in selected_scores.items():
    raw_score = raw_alpha001.loc[latest_date, symbol]
    confidence = (
        0.35 * recent_ic_score
        + 0.30 * stock_score_scaled.get(symbol, 0.5)
        + 0.20 * float(score > 0)
        + 0.15 * float(regime_allowed)
    )
    candidate_rows.append({
        "date": latest_date,
        "symbol": symbol,
        "alpha": "alpha001",
        "suggested_direction": "long" if not final_params.long_short else "long_leg_candidate",
        "target_weight_equal_weight": 1.0 / len(selected_scores) if len(selected_scores) else np.nan,
        "raw_alpha001_score": raw_score,
        "oriented_signal_score": score,
        "signal_orientation": final_orientation_label,
        "horizon_days": DEFAULT_HORIZON,
        "recent_train_raw_mean_ic": final_train_ic,
        "recent_ic_score": recent_ic_score,
        "stock_score": stock_score_scaled.get(symbol, 0.5),
        "confidence_v0": confidence,
        "trend_regime": latest_regime.get("trend_regime"),
        "vol_regime": latest_regime.get("vol_regime"),
        "regime_filter": final_params.regime_filter,
        "reason": (
            f"passes final Alpha#1 rule: top_n={final_params.top_n}, min_signal={final_params.min_signal}, "
            f"rebalance_every={final_params.rebalance_every}, orientation={final_orientation_label}, "
            f"regime_allowed={regime_allowed}"
        ),
    })

latest_candidates = pd.DataFrame(candidate_rows).sort_values("confidence_v0", ascending=False) if candidate_rows else pd.DataFrame()
stock_ranking.to_csv(ARTIFACT_DIR / "stock_level_alpha_report.csv")
final_train_results.head(25).to_csv(ARTIFACT_DIR / "final_recent_training_leaderboard.csv", index=False)
latest_candidates.to_csv(ARTIFACT_DIR / "latest_alpha001_candidates.csv", index=False)

print("Latest signal date:", latest_date.date())
print("Final recent-training orientation:", final_orientation_label)
print("Final params:", final_params)
print("Current regime:", latest_regime[["trend_regime", "vol_regime"]].to_dict())
print("Regime allowed today:", regime_allowed)
print("Recent training leaderboard")
display(final_train_results.head(10))
print("Top stock compatibility scores")
display(stock_ranking.head(15))
print("Latest Alpha#1 candidate table")
display(latest_candidates)

## 10. Alpha#1 Implementation Audit

Before investigating regimes, we first check that we are not fooling ourselves with a formula convention. Alpha#1 has two implementation choices that matter in practice:

- whether the price input should be raw close or adjusted close,
- how `ts_argmax(x, 5)` is interpreted: oldest-to-newest position or days-since-maximum.

This section reuses in-memory notebook variables when available. In a fresh kernel, it loads the cached real-data artifacts and does only the light diagnostics below; it does not redownload data or rerun the full grid search.

In [2]:
# Reuse the current notebook state when possible; otherwise load cached real-data artifacts.
try:
    _ = oriented_alpha001
    print("Using in-memory notebook variables.")
except NameError:
    print("Loading cached real-data artifacts; no yfinance download and no full grid recomputation.")
    DATA_DIR = Path("research/data/alpha001_nifty50")
    ARTIFACT_DIR = Path("research/artifacts/alpha001_research_to_alpha")
    DEFAULT_HORIZON = 5
    FORWARD_HORIZONS = [1, 3, 5, 10]
    ANNUALIZATION = 252
    COST_BPS_PER_TURNOVER = 10.0

    def _read_frame(path: Path) -> pd.DataFrame:
        frame = pd.read_csv(path, index_col=0, parse_dates=True)
        frame.index = pd.to_datetime(frame.index).tz_localize(None)
        return frame.sort_index()

    raw_alpha001 = _read_frame(ARTIFACT_DIR / "raw_alpha001_scores.csv")
    oriented_alpha001 = _read_frame(ARTIFACT_DIR / "oriented_alpha001_scores.csv")
    return_price_all = _read_frame(DATA_DIR / "adj_close.csv")
    raw_close_all = _read_frame(DATA_DIR / "close.csv")
    volume_all = _read_frame(DATA_DIR / "volume.csv")
    nifty_close = _read_frame(DATA_DIR / "nifty_close.csv").iloc[:, 0].rename("nifty_close")
    usable_symbols = list(oriented_alpha001.columns)
    return_price = return_price_all.reindex(columns=usable_symbols)
    raw_close = raw_close_all.reindex(columns=usable_symbols)
    volume = volume_all.reindex(columns=usable_symbols)
    returns = return_price.pct_change(fill_method=None)
    forward_returns = {h: return_price.shift(-h).div(return_price).sub(1.0) for h in FORWARD_HORIZONS}
    next_session_returns = return_price.pct_change(fill_method=None).shift(-1)


def _rank_corr(x: pd.Series, y: pd.Series) -> float:
    pair = pd.concat([x.astype(float), y.astype(float)], axis=1).dropna()
    if len(pair) < 3:
        return np.nan
    return pair.iloc[:, 0].rank().corr(pair.iloc[:, 1].rank(), method="pearson")


def cross_sectional_corr_by_date(signal: pd.DataFrame, future: pd.DataFrame, method: str = "spearman", min_names: int = 10) -> pd.Series:
    rows = []
    for date in signal.index.intersection(future.index):
        s = signal.loc[date]
        f = future.loc[date]
        valid = s.notna() & f.notna()
        if valid.sum() < min_names:
            rows.append((date, np.nan))
            continue
        x = s[valid].astype(float)
        y = f[valid].astype(float)
        if method == "spearman":
            x = x.rank()
            y = y.rank()
        rows.append((date, x.corr(y, method="pearson")))
    return pd.Series(dict(rows)).sort_index().rename("rank_ic")


def signed_power(frame: pd.DataFrame, power: float) -> pd.DataFrame:
    return np.sign(frame) * (frame.abs() ** power)


def rank_cross_sectional(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.rank(axis=1, pct=True)


def ts_argmax_position(frame: pd.DataFrame, window: int, convention: str) -> pd.DataFrame:
    def position(values: np.ndarray) -> float:
        if np.isnan(values).all():
            return np.nan
        idx = int(np.nanargmax(values))
        if convention == "oldest_to_newest":
            return float(idx + 1)
        if convention == "days_since_max":
            return float(window - idx)
        raise ValueError(f"Unknown convention: {convention}")
    return frame.rolling(window, min_periods=window).apply(position, raw=True)


def compute_alpha001_variant(close: pd.DataFrame, return_frame: pd.DataFrame, convention: str) -> pd.DataFrame:
    vol20 = return_frame.rolling(20, min_periods=20).std()
    condition_value = close.where(~return_frame.lt(0), vol20)
    transformed = signed_power(condition_value, 2)
    argmax5 = ts_argmax_position(transformed, 5, convention=convention)
    return rank_cross_sectional(argmax5) - 0.5


def summarize_ic(signal: pd.DataFrame, fwd_by_horizon: dict[int, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for horizon, fwd in fwd_by_horizon.items():
        rank_ic = cross_sectional_corr_by_date(signal, fwd, method="spearman")
        rows.append({
            "horizon_days": horizon,
            "mean_rank_ic": rank_ic.mean(),
            "median_rank_ic": rank_ic.median(),
            "positive_rank_ic_rate": rank_ic.dropna().gt(0).mean(),
            "observations": rank_ic.notna().sum(),
        })
    return pd.DataFrame(rows).set_index("horizon_days")


sanity = pd.Series({
    "alpha_start": oriented_alpha001.index.min(),
    "alpha_end": oriented_alpha001.index.max(),
    "symbols": oriented_alpha001.shape[1],
    "non_null_scores": int(oriented_alpha001.notna().sum().sum()),
    "min_score": oriented_alpha001.min().min(),
    "max_score": oriented_alpha001.max().max(),
    "median_daily_coverage": oriented_alpha001.notna().sum(axis=1).median(),
})
display(sanity.to_frame("implementation_sanity"))

Loading cached real-data artifacts; no yfinance download and no full grid recomputation.


,implementation_sanity
alpha_start,2018-01-01 00:00:00
alpha_end,2026-05-19 00:00:00
symbols,48
non_null_scores,96469
min_score,-0.500000
max_score,0.479167
median_daily_coverage,47.000000


In [3]:
variant_specs = {
    "adjusted_close_oldest_to_newest": (return_price, return_price.pct_change(fill_method=None), "oldest_to_newest"),
    "adjusted_close_days_since_max": (return_price, return_price.pct_change(fill_method=None), "days_since_max"),
    "raw_close_oldest_to_newest": (raw_close, raw_close.pct_change(fill_method=None), "oldest_to_newest"),
    "raw_close_days_since_max": (raw_close, raw_close.pct_change(fill_method=None), "days_since_max"),
}

variant_rows = []
variant_signals = {}
base_stack = oriented_alpha001.stack().rename("base_oriented")

for name, (price_input, return_input, convention) in variant_specs.items():
    signal = compute_alpha001_variant(price_input, return_input, convention=convention)
    raw_summary = summarize_ic(signal, forward_returns)
    default_raw_ic = raw_summary.loc[DEFAULT_HORIZON, "mean_rank_ic"]
    direction = 1 if default_raw_ic >= 0 else -1
    oriented_variant = direction * signal
    oriented_summary = summarize_ic(oriented_variant, forward_returns)
    variant_signals[name] = oriented_variant
    pair = pd.concat([base_stack, oriented_variant.stack().rename("variant")], axis=1).dropna()
    variant_rows.append({
        "variant": name,
        "orientation": "raw" if direction == 1 else "inverted",
        "default_raw_mean_ic": default_raw_ic,
        "oriented_1d_ic": oriented_summary.loc[1, "mean_rank_ic"],
        "oriented_3d_ic": oriented_summary.loc[3, "mean_rank_ic"],
        "oriented_5d_ic": oriented_summary.loc[5, "mean_rank_ic"],
        "oriented_10d_ic": oriented_summary.loc[10, "mean_rank_ic"],
        "positive_5d_ic_rate": oriented_summary.loc[5, "positive_rank_ic_rate"],
        "rank_corr_to_current_oriented": _rank_corr(pair["base_oriented"], pair["variant"]),
        "observations_5d": oriented_summary.loc[5, "observations"],
    })

implementation_audit = pd.DataFrame(variant_rows).sort_values("oriented_5d_ic", ascending=False)
implementation_audit.to_csv(ARTIFACT_DIR / "implementation_variant_audit.csv", index=False)
display(implementation_audit)

,variant,orientation,default_raw_mean_ic,oriented_1d_ic,oriented_3d_ic,oriented_5d_ic,oriented_10d_ic,positive_5d_ic_rate,rank_corr_to_current_oriented,observations_5d
0,adjusted_close_oldest_to_newest,inverted,-0.018053,0.021877,0.021003,0.018053,0.014581,0.534074,1.000000,2042
1,adjusted_close_days_since_max,raw,0.018053,0.021877,0.021003,0.018053,0.014581,0.534074,0.999462,2042
2,raw_close_oldest_to_newest,inverted,-0.017971,0.021583,0.020793,0.017971,0.014818,0.532624,0.993639,2042
3,raw_close_days_since_max,raw,0.017971,0.021583,0.020793,0.017971,0.014818,0.532624,0.993100,2042


## 11. Date-Level Conditions: When Does Alpha#1 Work?

Now we condition the oriented alpha on market state. These features are lagged where they come from market data, so the bucket label at date `t` only uses information available before the forward-return window.

We look at three outputs in each pocket:

- **Rank IC**: cross-sectional predictive quality.
- **Top-bottom spread**: top Alpha#1 bucket minus bottom Alpha#1 bucket at the default horizon.
- **Research-rule return**: a single cheap weekly rule selected from the existing leaderboard, used only as a sanity P&L lens.

In [4]:
def qcut_labels(series: pd.Series, labels: list[str]) -> pd.Series:
    valid = series.replace([np.inf, -np.inf], np.nan).dropna()
    out = pd.Series(index=series.index, dtype="object")
    if valid.nunique() < 2:
        out.loc[valid.index] = "single_bucket"
        return out
    try:
        bucketed = pd.qcut(valid, q=len(labels), labels=labels, duplicates="drop")
    except ValueError:
        out.loc[valid.index] = "single_bucket"
        return out
    out.loc[bucketed.index] = bucketed.astype(str)
    return out


def bucket_returns(signal: pd.DataFrame, future: pd.DataFrame, top_n: int) -> pd.DataFrame:
    rows = []
    for date in signal.index.intersection(future.index):
        scores = signal.loc[date].dropna()
        rets = future.loc[date].dropna()
        common = scores.index.intersection(rets.index)
        if len(common) < top_n * 2:
            continue
        scores = scores[common].sort_values(ascending=False)
        rets = rets[common]
        top = scores.head(top_n).index
        bottom = scores.tail(top_n).index
        rows.append({
            "date": date,
            "top_return": rets[top].mean(),
            "bottom_return": rets[bottom].mean(),
            "spread_return": rets[top].mean() - rets[bottom].mean(),
        })
    return pd.DataFrame(rows).set_index("date") if rows else pd.DataFrame(columns=["top_return", "bottom_return", "spread_return"])


def performance_metrics(returns: pd.Series, periods_per_year: int = 252) -> dict:
    r = returns.dropna()
    if r.empty:
        return {"ann_return": np.nan, "ann_vol": np.nan, "sharpe": np.nan, "max_drawdown": np.nan}
    equity = (1 + r).cumprod()
    years = len(r) / periods_per_year
    ann_return = equity.iloc[-1] ** (1 / years) - 1 if years > 0 and equity.iloc[-1] > 0 else np.nan
    ann_vol = r.std(ddof=0) * math.sqrt(periods_per_year)
    sharpe = (r.mean() * periods_per_year) / ann_vol if ann_vol and ann_vol > 0 else np.nan
    drawdown = equity / equity.cummax() - 1
    return {"ann_return": ann_return, "ann_vol": ann_vol, "sharpe": sharpe, "max_drawdown": drawdown.min()}


def make_regime_frame(index_close: pd.Series) -> pd.DataFrame:
    n = index_close.dropna().copy()
    sma50 = n.rolling(50, min_periods=50).mean()
    sma200 = n.rolling(200, min_periods=200).mean()
    index_ret = n.pct_change(fill_method=None)
    vol20 = index_ret.rolling(20, min_periods=20).std() * math.sqrt(252)
    vol_history = vol20.shift(1)
    vol_q_low = vol_history.rolling(252, min_periods=100).quantile(0.33)
    vol_q_high = vol_history.rolling(252, min_periods=100).quantile(0.67)
    trend = pd.Series("sideways", index=n.index, dtype="object")
    trend[(n > sma50) & (sma50 > sma200)] = "bull"
    trend[(n < sma50) & (sma50 < sma200)] = "bear"
    trend[sma50.isna() | sma200.isna()] = "unknown"
    vol_regime = pd.Series("medium_vol", index=n.index, dtype="object")
    vol_regime[vol20 <= vol_q_low] = "low_vol"
    vol_regime[vol20 >= vol_q_high] = "high_vol"
    vol_regime[vol20.isna() | vol_q_low.isna() | vol_q_high.isna()] = "unknown"
    return pd.DataFrame({"trend_regime": trend, "vol_regime": vol_regime, "nifty_vol20_ann": vol20})


def build_weekly_top_rule_returns(signal: pd.DataFrame, next_returns: pd.DataFrame, top_n: int, min_signal: float, rebalance_every: int, cost_bps: float) -> pd.Series:
    weights = pd.DataFrame(0.0, index=signal.index, columns=signal.columns)
    last_weights = pd.Series(0.0, index=signal.columns)
    for i, date in enumerate(signal.index):
        if i % rebalance_every != 0:
            weights.loc[date] = last_weights
            continue
        scores = signal.loc[date].dropna().sort_values(ascending=False)
        scores = scores[scores >= min_signal]
        if len(scores) < top_n:
            last_weights = pd.Series(0.0, index=signal.columns)
        else:
            last_weights = pd.Series(0.0, index=signal.columns)
            last_weights.loc[scores.head(top_n).index] = 1.0 / top_n
        weights.loc[date] = last_weights
    aligned_returns = next_returns.reindex_like(weights)
    gross = weights.mul(aligned_returns).sum(axis=1)
    turnover = weights.diff().abs().sum(axis=1, min_count=1).fillna(weights.abs().sum(axis=1))
    return gross - turnover * (cost_bps / 10000.0)


rank_ic_default = cross_sectional_corr_by_date(oriented_alpha001, forward_returns[DEFAULT_HORIZON], method="spearman")
default_spread = bucket_returns(oriented_alpha001, forward_returns[DEFAULT_HORIZON], top_n=10)["spread_return"]

leaderboard_path = ARTIFACT_DIR / "rule_search_leaderboard.csv"
if leaderboard_path.exists():
    first_rule = pd.read_csv(leaderboard_path).iloc[0]
    rule_top_n = int(first_rule["top_n"])
    rule_min_signal = float(first_rule["min_signal"])
    rule_rebalance = int(first_rule["rebalance_every"])
else:
    rule_top_n, rule_min_signal, rule_rebalance = 5, 0.30, 5
research_rule_returns = build_weekly_top_rule_returns(
    oriented_alpha001,
    next_session_returns,
    top_n=rule_top_n,
    min_signal=rule_min_signal,
    rebalance_every=rule_rebalance,
    cost_bps=COST_BPS_PER_TURNOVER,
).rename("research_rule_return")

regime_frame = make_regime_frame(nifty_close)
market_ret = nifty_close.pct_change(fill_method=None)
feature_frame = pd.DataFrame(index=oriented_alpha001.index)
feature_frame["nifty_trend_regime"] = regime_frame["trend_regime"]
feature_frame["nifty_vol_regime"] = regime_frame["vol_regime"]
feature_frame["market_20d_momentum"] = qcut_labels(nifty_close.pct_change(20, fill_method=None).shift(1), ["weak", "middle", "strong"])
feature_frame["market_63d_momentum"] = qcut_labels(nifty_close.pct_change(63, fill_method=None).shift(1), ["weak", "middle", "strong"])
feature_frame["market_20d_volatility"] = qcut_labels(market_ret.rolling(20, min_periods=20).std().shift(1) * math.sqrt(252), ["low", "middle", "high"])
feature_frame["breadth_20d"] = qcut_labels(returns.gt(0).mean(axis=1).rolling(20, min_periods=20).mean().shift(1), ["weak", "middle", "broad"])
feature_frame["dispersion_20d"] = qcut_labels(returns.std(axis=1).rolling(20, min_periods=20).mean().shift(1) * math.sqrt(252), ["low", "middle", "high"])
volume_ratio = volume.div(volume.rolling(20, min_periods=20).mean()).replace([np.inf, -np.inf], np.nan)
feature_frame["volume_intensity_20d"] = qcut_labels(volume_ratio.median(axis=1).rolling(5, min_periods=5).mean().shift(1), ["quiet", "normal", "heavy"])
signal_width = oriented_alpha001.quantile(0.90, axis=1) - oriented_alpha001.quantile(0.10, axis=1)
feature_frame["signal_cross_section_width"] = qcut_labels(signal_width, ["narrow", "middle", "wide"])


def conditional_report(labels: pd.Series, condition_name: str) -> pd.DataFrame:
    df = pd.concat([
        labels.rename("bucket"),
        rank_ic_default.rename("rank_ic"),
        default_spread.rename("top_bottom_spread"),
        research_rule_returns.rename("research_rule_return"),
    ], axis=1).dropna(subset=["bucket"])
    rows = []
    for bucket, group in df.groupby("bucket", dropna=True):
        perf = performance_metrics(group["research_rule_return"])
        rows.append({
            "condition": condition_name,
            "bucket": bucket,
            "days": len(group),
            "mean_rank_ic": group["rank_ic"].mean(),
            "positive_ic_rate": group["rank_ic"].dropna().gt(0).mean(),
            "mean_top_bottom_spread": group["top_bottom_spread"].mean(),
            "spread_hit_rate": group["top_bottom_spread"].dropna().gt(0).mean(),
            "rule_ann_return": perf["ann_return"],
            "rule_sharpe": perf["sharpe"],
            "rule_max_drawdown": perf["max_drawdown"],
        })
    return pd.DataFrame(rows)

conditional_results = pd.concat(
    [conditional_report(feature_frame[col], col) for col in feature_frame.columns],
    ignore_index=True,
)
conditional_results = conditional_results.sort_values(["condition", "mean_rank_ic"], ascending=[True, False])
conditional_results.to_csv(ARTIFACT_DIR / "conditional_date_regime_report.csv", index=False)

display(conditional_results)
print("Best IC pockets")
display(conditional_results.query("days >= 126").sort_values("mean_rank_ic", ascending=False).head(12))
print("Worst IC pockets")
display(conditional_results.query("days >= 126").sort_values("mean_rank_ic", ascending=True).head(12))

,condition,bucket,days,mean_rank_ic,positive_ic_rate,mean_top_bottom_spread,spread_hit_rate,rule_ann_return,rule_sharpe,rule_max_drawdown
18,breadth_20d,middle,687,0.020050,0.537118,0.001357,0.537118,-0.053136,-0.226437,-0.317206
17,breadth_20d,broad,679,0.018472,0.546392,0.000812,0.505155,0.341501,1.905173,-0.126307
19,breadth_20d,weak,683,0.015609,0.534407,0.001289,0.518302,0.459875,1.906928,-0.104962
20,dispersion_20d,high,683,0.024570,0.554905,0.001526,0.535871,0.478343,1.867985,-0.130074
22,dispersion_20d,middle,682,0.015011,0.524927,0.001063,0.508798,0.188795,1.088101,-0.176254
21,dispersion_20d,low,683,0.014548,0.538799,0.000870,0.516837,0.053261,0.428859,-0.222188
9,market_20d_momentum,strong,679,0.023999,0.564065,0.000555,0.516937,0.322321,1.678230,-0.165509
8,market_20d_momentum,middle,679,0.016305,0.536082,0.001207,0.515464,0.047717,0.383015,-0.173721
10,market_20d_momentum,weak,680,0.015182,0.522059,0.001760,0.530882,0.331345,1.427251,-0.149492
14,market_20d_volatility,high,650,0.026256,0.540000,0.001942,0.526154,0.438317,1.792423,-0.120477


Best IC pockets


,condition,bucket,days,mean_rank_ic,positive_ic_rate,mean_top_bottom_spread,spread_hit_rate,rule_ann_return,rule_sharpe,rule_max_drawdown
0,nifty_trend_regime,bear,200,0.034116,0.550000,0.003603,0.515000,0.570088,2.017387,-0.106056
3,nifty_trend_regime,unknown,199,0.032481,0.522613,0.002074,0.482412,-0.042955,-0.136910,-0.166198
24,volume_intensity_20d,normal,681,0.026582,0.565345,0.001355,0.524229,0.252151,1.320300,-0.207483
14,market_20d_volatility,high,650,0.026256,0.540000,0.001942,0.526154,0.438317,1.792423,-0.120477
13,market_63d_momentum,weak,665,0.024838,0.542857,0.001880,0.526316,0.486291,1.941828,-0.146479
20,dispersion_20d,high,683,0.024570,0.554905,0.001526,0.535871,0.478343,1.867985,-0.130074
9,market_20d_momentum,strong,679,0.023999,0.564065,0.000555,0.516937,0.322321,1.678230,-0.165509
5,nifty_vol_regime,low_vol,688,0.021478,0.581395,0.000899,0.539244,0.181863,1.132125,-0.123571
26,signal_cross_section_width,middle,678,0.020082,0.539823,0.000928,0.508850,-0.041082,-0.156985,-0.243539
18,breadth_20d,middle,687,0.020050,0.537118,0.001357,0.537118,-0.053136,-0.226437,-0.317206


Worst IC pockets


,condition,bucket,days,mean_rank_ic,positive_ic_rate,mean_top_bottom_spread,spread_hit_rate,rule_ann_return,rule_sharpe,rule_max_drawdown
11,market_63d_momentum,middle,665,0.010916,0.524812,0.000663,0.508271,-0.008302,0.034069,-0.175708
23,volume_intensity_20d,heavy,682,0.011152,0.520528,0.000864,0.520528,0.293518,1.394143,-0.152534
6,nifty_vol_regime,medium_vol,592,0.011888,0.511824,0.000212,0.486486,0.205498,1.158201,-0.164994
16,market_20d_volatility,middle,650,0.012666,0.533846,0.001370,0.518462,0.142907,0.826960,-0.136222
2,nifty_trend_regime,sideways,756,0.014224,0.526455,0.000068,0.514550,0.341746,1.612123,-0.183747
21,dispersion_20d,low,683,0.014548,0.538799,0.000870,0.516837,0.053261,0.428859,-0.222188
1,nifty_trend_regime,bull,908,0.014887,0.539648,0.001333,0.522026,0.137698,0.910163,-0.151947
22,dispersion_20d,middle,682,0.015011,0.524927,0.001063,0.508798,0.188795,1.088101,-0.176254
10,market_20d_momentum,weak,680,0.015182,0.522059,0.001760,0.530882,0.331345,1.427251,-0.149492
19,breadth_20d,weak,683,0.015609,0.534407,0.001289,0.518302,0.459875,1.906928,-0.104962


## 12. Stock-Level Filters: Volume, Momentum, Volatility

Date-level regimes tell us about the tape. This section asks a different question: among names selected by Alpha#1, which *stock-level* conditions make the selection more or less useful?

For each day, we take the top and bottom Alpha#1 buckets and attach lagged stock-level features: short momentum, medium momentum, short reversal, realized volatility, volume intensity, and dollar volume.

In [5]:
asset_features = {
    "stock_mom_20d": return_price.pct_change(20, fill_method=None).shift(1),
    "stock_mom_63d": return_price.pct_change(63, fill_method=None).shift(1),
    "stock_reversal_5d": return_price.pct_change(5, fill_method=None).shift(1),
    "stock_vol_20d": returns.rolling(20, min_periods=20).std().shift(1) * math.sqrt(252),
    "stock_volume_intensity": volume.div(volume.rolling(20, min_periods=20).mean()).shift(1).replace([np.inf, -np.inf], np.nan),
    "stock_adv_20d": raw_close.mul(volume).rolling(20, min_periods=20).mean().shift(1),
}


def selected_name_records(signal: pd.DataFrame, future: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    rows = []
    for date in signal.index.intersection(future.index):
        scores = signal.loc[date].dropna().sort_values(ascending=False)
        rets = future.loc[date].dropna()
        common = scores.index.intersection(rets.index)
        if len(common) < top_n * 2:
            continue
        scores = scores[common]
        for side, names in {"top": scores.head(top_n).index, "bottom": scores.tail(top_n).index}.items():
            for symbol in names:
                row = {
                    "date": date,
                    "symbol": symbol,
                    "side": side,
                    "oriented_alpha001": scores.loc[symbol],
                    "future_return": rets.loc[symbol],
                }
                for feature_name, feature_values in asset_features.items():
                    row[feature_name] = feature_values.reindex(index=[date], columns=[symbol]).iloc[0, 0]
                rows.append(row)
    return pd.DataFrame(rows)

selection_records = selected_name_records(oriented_alpha001, forward_returns[DEFAULT_HORIZON], top_n=10)
selection_records.to_csv(ARTIFACT_DIR / "selected_name_feature_records.csv", index=False)

feature_rows = []
for feature_name in asset_features:
    work = selection_records.dropna(subset=[feature_name, "future_return"]).copy()
    if work[feature_name].nunique() < 3:
        continue
    work["feature_bucket"] = pd.qcut(work[feature_name], 3, labels=["low", "middle", "high"], duplicates="drop").astype(str)
    for (side, bucket), group in work.groupby(["side", "feature_bucket"]):
        feature_rows.append({
            "feature": feature_name,
            "side": side,
            "bucket": bucket,
            "rows": len(group),
            "mean_future_return": group["future_return"].mean(),
            "hit_rate": group["future_return"].gt(0).mean(),
            "median_alpha_score": group["oriented_alpha001"].median(),
        })

selected_feature_report = pd.DataFrame(feature_rows).sort_values(["feature", "side", "mean_future_return"], ascending=[True, True, False])
selected_feature_report.to_csv(ARTIFACT_DIR / "selected_name_feature_report.csv", index=False)
display(selected_feature_report)

print("Top-bucket stock conditions with best forward returns")
display(selected_feature_report.query("side == 'top' and rows >= 500").sort_values("mean_future_return", ascending=False).head(12))
print("Top-bucket stock conditions with weakest forward returns")
display(selected_feature_report.query("side == 'top' and rows >= 500").sort_values("mean_future_return", ascending=True).head(12))

,feature,side,bucket,rows,mean_future_return,hit_rate,median_alpha_score
32,stock_adv_20d,bottom,middle,6776,0.004996,0.540289,-0.333333
31,stock_adv_20d,bottom,low,6899,0.002637,0.514857,-0.336957
30,stock_adv_20d,bottom,high,6728,0.002076,0.511593,-0.329787
35,stock_adv_20d,top,middle,6828,0.004888,0.547598,0.364583
34,stock_adv_20d,top,low,6706,0.004195,0.543096,0.364583
33,stock_adv_20d,top,high,6876,0.004101,0.544503,0.364583
2,stock_mom_20d,bottom,middle,6825,0.004408,0.538755,-0.333333
0,stock_mom_20d,bottom,high,7672,0.003017,0.513947,-0.329787
1,stock_mom_20d,bottom,low,5913,0.002150,0.513614,-0.333333
3,stock_mom_20d,top,high,5935,0.005628,0.549958,0.364583


Top-bucket stock conditions with best forward returns


,feature,side,bucket,rows,mean_future_return,hit_rate,median_alpha_score
21,stock_vol_20d,top,high,7017,0.006556,0.548953,0.358696
3,stock_mom_20d,top,high,5935,0.005628,0.549958,0.364583
15,stock_reversal_5d,top,high,4418,0.005277,0.548212,0.347826
10,stock_mom_63d,top,low,6982,0.005245,0.550845,0.364583
9,stock_mom_63d,top,high,6235,0.004930,0.543545,0.369565
35,stock_adv_20d,top,middle,6828,0.004888,0.547598,0.364583
29,stock_volume_intensity,top,middle,6924,0.004815,0.549393,0.364583
16,stock_reversal_5d,top,low,8333,0.004527,0.554902,0.364583
27,stock_volume_intensity,top,high,6017,0.004239,0.548945,0.364583
34,stock_adv_20d,top,low,6706,0.004195,0.543096,0.364583


Top-bucket stock conditions with weakest forward returns


,feature,side,bucket,rows,mean_future_return,hit_rate,median_alpha_score
22,stock_vol_20d,top,low,6593,0.002775,0.535113,0.364583
11,stock_mom_63d,top,middle,6753,0.003054,0.542574,0.364583
5,stock_mom_20d,top,middle,6781,0.003612,0.543873,0.364583
23,stock_vol_20d,top,middle,6800,0.003737,0.550735,0.364583
17,stock_reversal_5d,top,middle,7659,0.003744,0.532576,0.369565
33,stock_adv_20d,top,high,6876,0.004101,0.544503,0.364583
28,stock_volume_intensity,top,low,7469,0.004132,0.537957,0.364583
4,stock_mom_20d,top,low,7694,0.004134,0.542371,0.364583
34,stock_adv_20d,top,low,6706,0.004195,0.543096,0.364583
27,stock_volume_intensity,top,high,6017,0.004239,0.548945,0.364583


## 13. Reading The Conditional Diagnostics

The key question is no longer whether Alpha#1 is a strong standalone alpha on average. It is not. The useful question is whether it becomes a good *conditional* alpha or filter.

Use the new reports this way:

- If implementation variants all point to the same weak IC, the weakness is not just a coding convention.
- If IC and spread improve in specific market buckets, Alpha#1 should be regime-gated.
- If selected-name returns improve in specific stock buckets, Alpha#1 should be combined with stock filters such as momentum, volatility, volume intensity, or liquidity.
- If IC improves but P&L does not, turnover/costs are eating the signal and the rule needs slower rebalancing or thresholding.
- If top and bottom buckets both rise in bull regimes, the alpha may be useful for ranking longs but not as a market-neutral spread.

## 14. NIFTY 500 High-Volatility Universe

This section expands the research parent universe from the current NIFTY50-style basket to the current NIFTY 500 constituent list. It then builds a dynamic high-volatility research basket.

The math rule is important: **universe membership is decided first**, using only lagged/known data. Alpha#1 ranks are recomputed inside that active universe afterward. This avoids corrupting the cross-sectional ranking when names enter or leave.

Primary universe rule:

- parent: current NIFTY 500 constituents,
- volatility score: lagged 20-day realized volatility,
- reconstitution: weekly,
- basket: top 100,
- retention buffer: keep existing names if still inside top 125,
- moderate filters: minimum 126 days of history, price >= INR 20, 20-day average rupee volume >= INR 5 crore.

In [ ]:
import time
from io import StringIO

import requests

try:
    import yfinance as yf
except ImportError as exc:
    raise ImportError("Install yfinance first: .venv/bin/python -m pip install -r requirements.txt") from exc

try:
    ARTIFACT_DIR
except NameError:
    ARTIFACT_DIR = Path("research/artifacts/alpha001_research_to_alpha")
try:
    START_DATE
except NameError:
    START_DATE = "2018-01-01"
try:
    END_DATE
except NameError:
    END_DATE = None
try:
    ANNUALIZATION
except NameError:
    ANNUALIZATION = 252
try:
    COST_BPS_PER_TURNOVER
except NameError:
    COST_BPS_PER_TURNOVER = 10.0
try:
    DEFAULT_HORIZON
except NameError:
    DEFAULT_HORIZON = 5
try:
    FORWARD_HORIZONS
except NameError:
    FORWARD_HORIZONS = [1, 3, 5, 10]

NIFTY500_DATA_DIR = Path("research/data/nifty500_high_vol")
NIFTY500_DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

NIFTY500_CONSTITUENT_PATH = NIFTY500_DATA_DIR / "nifty500_constituents.csv"
NIFTY500_DOWNLOAD_REPORT_PATH = ARTIFACT_DIR / "nifty500_download_report.csv"
DYNAMIC_HIGH_VOL_REPORT_PATH = ARTIFACT_DIR / "dynamic_high_vol_universe_report.csv"
DYNAMIC_HIGH_VOL_MASK_PATH = ARTIFACT_DIR / "dynamic_high_vol_universe_mask_top100.csv"

NIFTY500_FIELD_FILES = {
    "open": NIFTY500_DATA_DIR / "open.csv",
    "high": NIFTY500_DATA_DIR / "high.csv",
    "low": NIFTY500_DATA_DIR / "low.csv",
    "close": NIFTY500_DATA_DIR / "close.csv",
    "adj_close": NIFTY500_DATA_DIR / "adj_close.csv",
    "volume": NIFTY500_DATA_DIR / "volume.csv",
}

NIFTY500_CONSTITUENT_URLS = [
    "https://www.niftyindices.com/IndexConstituent/ind_nifty500list.csv",
    "https://archives.nseindia.com/content/indices/ind_nifty500list.csv",
]

HIGH_VOL_BASKET_SIZE = 100
HIGH_VOL_BUFFER_SIZE = 125
HIGH_VOL_LOOKBACK = 20
HIGH_VOL_MIN_HISTORY = 126
HIGH_VOL_MIN_PRICE = 20.0
HIGH_VOL_MIN_ADV_RUPEES = 50_000_000.0
HIGH_VOL_REBALANCE = "weekly"
NIFTY500_DOWNLOAD_CHUNK_SIZE = 75
REFRESH_NIFTY500_CONSTITUENTS = False
REFRESH_NIFTY500_PRICES = False


def read_frame(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, index_col=0, parse_dates=True)
    frame.index = pd.to_datetime(frame.index).tz_localize(None)
    return frame.sort_index()


def write_frame(frame: pd.DataFrame | pd.Series, path: Path) -> None:
    out = frame.to_frame() if isinstance(frame, pd.Series) else frame
    out.to_csv(path)


def parse_constituent_csv(text: str, source_url: str) -> pd.DataFrame:
    frame = pd.read_csv(StringIO(text))
    frame.columns = [str(c).strip() for c in frame.columns]
    symbol_col = None
    for col in frame.columns:
        if col.strip().lower() == "symbol":
            symbol_col = col
            break
    if symbol_col is None:
        raise ValueError(f"Could not find Symbol column from {source_url}; columns={list(frame.columns)}")
    frame = frame.copy()
    frame["Symbol"] = frame[symbol_col].astype(str).str.strip().str.upper()
    frame = frame[frame["Symbol"].ne("") & frame["Symbol"].ne("NAN")]
    if "Series" in frame.columns:
        frame = frame[frame["Series"].fillna("EQ").astype(str).str.upper().eq("EQ")]
    frame["YahooTicker"] = frame["Symbol"].map(lambda symbol: f"{symbol}.NS")
    frame["source_url"] = source_url
    return frame.drop_duplicates("Symbol").sort_values("Symbol").reset_index(drop=True)


def load_nifty500_constituents(refresh: bool = False) -> pd.DataFrame:
    if NIFTY500_CONSTITUENT_PATH.exists() and not refresh:
        frame = pd.read_csv(NIFTY500_CONSTITUENT_PATH)
        print(f"Loaded cached NIFTY500 constituents: {len(frame)} symbols")
        return frame

    headers = {
        "User-Agent": "Mozilla/5.0 Alpha001 research notebook",
        "Accept": "text/csv,application/csv,text/plain,*/*",
        "Referer": "https://www.niftyindices.com/",
    }
    errors = []
    for url in NIFTY500_CONSTITUENT_URLS:
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            if "Symbol" not in response.text[:1000]:
                raise ValueError("response did not look like the constituent CSV")
            frame = parse_constituent_csv(response.text, url)
            frame.to_csv(NIFTY500_CONSTITUENT_PATH, index=False)
            print(f"Downloaded NIFTY500 constituents from {url}: {len(frame)} symbols")
            return frame
        except Exception as exc:
            errors.append({"url": url, "error": repr(exc)})
    pd.DataFrame(errors).to_csv(ARTIFACT_DIR / "nifty500_constituent_fetch_errors.csv", index=False)
    raise RuntimeError(
        "Could not fetch NIFTY500 constituents and no cache exists. "
        f"Place the official CSV at {NIFTY500_CONSTITUENT_PATH} and rerun this cell."
    )


def extract_yf_field(raw_data: pd.DataFrame, tickers: list[str], ticker_to_symbol: dict[str, str], field: str) -> pd.DataFrame:
    frames = {}
    for ticker in tickers:
        symbol = ticker_to_symbol[ticker]
        series = None
        if isinstance(raw_data.columns, pd.MultiIndex):
            for key in [(ticker, field), (field, ticker)]:
                if key in raw_data.columns:
                    series = raw_data[key]
                    break
        elif field in raw_data.columns and len(tickers) == 1:
            series = raw_data[field]
        if series is not None:
            frames[symbol] = pd.to_numeric(series, errors="coerce")
    out = pd.DataFrame(frames).sort_index()
    if not out.empty:
        out.index = pd.to_datetime(out.index).tz_localize(None)
        out = out.loc[~out.index.duplicated(keep="last")]
    return out


def download_nifty500_ohlcv(constituents: pd.DataFrame, refresh: bool = False, chunk_size: int = 75) -> dict[str, pd.DataFrame]:
    cache_ready = all(path.exists() for path in NIFTY500_FIELD_FILES.values())
    if cache_ready and not refresh:
        data = {name: read_frame(path) for name, path in NIFTY500_FIELD_FILES.items()}
        print(f"Loaded cached NIFTY500 OHLCV from {NIFTY500_DATA_DIR}")
        return data

    symbols = constituents["Symbol"].dropna().astype(str).tolist()
    tickers = [f"{symbol}.NS" for symbol in symbols]
    ticker_to_symbol = dict(zip(tickers, symbols))
    field_parts = {field: [] for field in NIFTY500_FIELD_FILES}
    report_rows = []

    for start in range(0, len(tickers), chunk_size):
        chunk_tickers = tickers[start:start + chunk_size]
        chunk_symbols = [ticker_to_symbol[ticker] for ticker in chunk_tickers]
        print(f"Downloading NIFTY500 chunk {start // chunk_size + 1}: {chunk_symbols[0]} ... {chunk_symbols[-1]} ({len(chunk_symbols)} names)")
        try:
            raw = yf.download(
                tickers=chunk_tickers,
                start=START_DATE,
                end=END_DATE,
                auto_adjust=False,
                group_by="ticker",
                threads=True,
                progress=False,
            )
            for field, yf_field in {
                "open": "Open",
                "high": "High",
                "low": "Low",
                "close": "Close",
                "adj_close": "Adj Close",
                "volume": "Volume",
            }.items():
                extracted = extract_yf_field(raw, chunk_tickers, ticker_to_symbol, yf_field)
                field_parts[field].append(extracted)
            close_part = extract_yf_field(raw, chunk_tickers, ticker_to_symbol, "Close")
            for symbol in chunk_symbols:
                non_null = int(close_part[symbol].notna().sum()) if symbol in close_part else 0
                report_rows.append({"symbol": symbol, "ticker": f"{symbol}.NS", "status": "ok" if non_null else "missing", "close_rows": non_null, "error": ""})
        except Exception as exc:
            for symbol in chunk_symbols:
                report_rows.append({"symbol": symbol, "ticker": f"{symbol}.NS", "status": "error", "close_rows": 0, "error": repr(exc)})
        time.sleep(0.25)

    data = {}
    for field, parts in field_parts.items():
        non_empty_parts = [part for part in parts if not part.empty]
        frame = pd.concat(non_empty_parts, axis=1) if non_empty_parts else pd.DataFrame()
        frame = frame.loc[:, ~frame.columns.duplicated()].sort_index()
        frame = frame.reindex(columns=symbols)
        write_frame(frame, NIFTY500_FIELD_FILES[field])
        data[field] = frame

    report = pd.DataFrame(report_rows)
    report.to_csv(NIFTY500_DOWNLOAD_REPORT_PATH, index=False)
    print("NIFTY500 download report")
    display(report["status"].value_counts(dropna=False).to_frame("symbols"))
    return data


nifty500_constituents = load_nifty500_constituents(refresh=REFRESH_NIFTY500_CONSTITUENTS)
nifty500_data = download_nifty500_ohlcv(
    nifty500_constituents,
    refresh=REFRESH_NIFTY500_PRICES,
    chunk_size=NIFTY500_DOWNLOAD_CHUNK_SIZE,
)

n500_open = nifty500_data["open"]
n500_high = nifty500_data["high"]
n500_low = nifty500_data["low"]
n500_close = nifty500_data["close"]
n500_adj_close = nifty500_data["adj_close"]
n500_volume = nifty500_data["volume"]

n500_summary = pd.Series({
    "parent_symbols": len(nifty500_constituents),
    "price_columns": n500_adj_close.shape[1],
    "first_date": n500_adj_close.index.min(),
    "last_date": n500_adj_close.index.max(),
    "symbols_with_any_adjusted_close": int(n500_adj_close.notna().any().sum()),
})
display(n500_summary.to_frame("nifty500_data"))

In [ ]:
def weekly_reconstitution_mask(index: pd.DatetimeIndex) -> pd.Series:
    weeks = pd.Series(index.to_period("W-FRI"), index=index)
    mask = weeks.ne(weeks.shift(1))
    if len(mask):
        mask.iloc[0] = True
    return mask


def build_dynamic_high_vol_universe(
    adjusted_close: pd.DataFrame,
    raw_close: pd.DataFrame,
    volume_frame: pd.DataFrame,
    basket_size: int = 100,
    buffer_size: int = 125,
    vol_lookback: int = 20,
    min_history: int = 126,
    min_price: float = 20.0,
    min_adv_rupees: float = 50_000_000.0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    returns_frame = adjusted_close.pct_change(fill_method=None)
    realized_vol = returns_frame.rolling(vol_lookback, min_periods=vol_lookback).std().shift(1) * math.sqrt(ANNUALIZATION)
    adv20 = raw_close.mul(volume_frame).rolling(20, min_periods=20).mean().shift(1)
    history_ok = adjusted_close.notna().rolling(min_history, min_periods=min_history).sum().shift(1).ge(min_history)
    price_ok = raw_close.shift(1).ge(min_price)
    eligible = history_ok & price_ok & adv20.ge(min_adv_rupees) & realized_vol.notna()

    membership = pd.DataFrame(False, index=adjusted_close.index, columns=adjusted_close.columns)
    reconstitute = weekly_reconstitution_mask(adjusted_close.index)
    current_members: list[str] = []
    rows = []

    for date in adjusted_close.index:
        if bool(reconstitute.loc[date]):
            scores = realized_vol.loc[date].where(eligible.loc[date]).dropna().sort_values(ascending=False)
            eligible_count = len(scores)
            buffer_names = set(scores.head(buffer_size).index)
            keepers = [name for name in current_members if name in buffer_names]
            additions = [name for name in scores.index if name not in set(keepers)]
            previous = set(current_members)
            current_members = (keepers + additions)[:basket_size]
            current = set(current_members)
            rows.append({
                "date": date,
                "eligible_count": eligible_count,
                "member_count": len(current_members),
                "entries": len(current - previous),
                "exits": len(previous - current),
                "entry_symbols": ",".join(sorted(current - previous)),
                "exit_symbols": ",".join(sorted(previous - current)),
            })
        if current_members:
            membership.loc[date, current_members] = True

    report = pd.DataFrame(rows)
    if not report.empty:
        report["membership_turnover_fraction"] = (report["entries"] + report["exits"]) / (2 * basket_size)
    return membership, report


high_vol_mask_100, high_vol_universe_report = build_dynamic_high_vol_universe(
    n500_adj_close,
    n500_close,
    n500_volume,
    basket_size=HIGH_VOL_BASKET_SIZE,
    buffer_size=HIGH_VOL_BUFFER_SIZE,
    vol_lookback=HIGH_VOL_LOOKBACK,
    min_history=HIGH_VOL_MIN_HISTORY,
    min_price=HIGH_VOL_MIN_PRICE,
    min_adv_rupees=HIGH_VOL_MIN_ADV_RUPEES,
)

high_vol_mask_100.astype(int).to_csv(DYNAMIC_HIGH_VOL_MASK_PATH)
high_vol_universe_report.to_csv(DYNAMIC_HIGH_VOL_REPORT_PATH, index=False)

universe_daily_count = high_vol_mask_100.sum(axis=1)
universe_sanity = pd.Series({
    "first_non_empty_date": universe_daily_count[universe_daily_count > 0].index.min() if universe_daily_count.gt(0).any() else pd.NaT,
    "last_date": universe_daily_count.index.max(),
    "median_member_count_after_warmup": universe_daily_count[universe_daily_count > 0].median(),
    "min_member_count_after_warmup": universe_daily_count[universe_daily_count > 0].min(),
    "max_member_count": universe_daily_count.max(),
    "weekly_reconstitutions": len(high_vol_universe_report),
    "average_weekly_membership_turnover": high_vol_universe_report["membership_turnover_fraction"].mean() if not high_vol_universe_report.empty else np.nan,
})
display(universe_sanity.to_frame("dynamic_high_vol_universe"))
display(high_vol_universe_report.tail(12))

fig, ax = plt.subplots(figsize=(12, 4))
universe_daily_count.plot(ax=ax, color="#174A7C")
ax.set_title("Dynamic high-volatility universe member count")
ax.set_ylabel("Members")
plt.show()

## 15. Alpha#1 on the Dynamic High-Volatility Basket

This section recomputes Alpha#1 inside three different research universes:

1. current NIFTY50-style baseline already studied earlier,
2. all eligible NIFTY500 names,
3. dynamic NIFTY500 top-100 high-volatility basket.

For the high-volatility basket, ranks are computed after applying the active membership mask, so a stock cannot influence the cross-section before it is eligible and selected into the basket.

In [ ]:
def ts_argmax_position(frame: pd.DataFrame, window: int, convention: str = "oldest_to_newest") -> pd.DataFrame:
    def position(values: np.ndarray) -> float:
        if np.isnan(values).all():
            return np.nan
        idx = int(np.nanargmax(values))
        if convention == "oldest_to_newest":
            return float(idx + 1)
        if convention == "days_since_max":
            return float(window - idx)
        raise ValueError(f"Unknown convention: {convention}")
    return frame.rolling(window, min_periods=window).apply(position, raw=True)


def signed_power(frame: pd.DataFrame, power: float) -> pd.DataFrame:
    return np.sign(frame) * (frame.abs() ** power)


def rank_cross_sectional(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.rank(axis=1, pct=True)


def compute_alpha001_inside_universe(close_frame: pd.DataFrame, return_frame: pd.DataFrame, active_mask: pd.DataFrame) -> pd.DataFrame:
    vol20 = return_frame.rolling(20, min_periods=20).std()
    condition_value = close_frame.where(~return_frame.lt(0), vol20)
    transformed = signed_power(condition_value, 2)
    argmax5 = ts_argmax_position(transformed, 5, convention="oldest_to_newest")
    active_argmax = argmax5.where(active_mask.reindex_like(argmax5).fillna(False))
    return rank_cross_sectional(active_argmax) - 0.5


def rank_corr(x: pd.Series, y: pd.Series) -> float:
    pair = pd.concat([x.astype(float), y.astype(float)], axis=1).dropna()
    if len(pair) < 3:
        return np.nan
    return pair.iloc[:, 0].rank().corr(pair.iloc[:, 1].rank(), method="pearson")


def cross_sectional_corr_by_date(signal: pd.DataFrame, future: pd.DataFrame, method: str = "spearman", min_names: int = 10) -> pd.Series:
    rows = []
    for date in signal.index.intersection(future.index):
        s = signal.loc[date]
        f = future.loc[date]
        valid = s.notna() & f.notna()
        if valid.sum() < min_names:
            rows.append((date, np.nan))
            continue
        x = s[valid].astype(float)
        y = f[valid].astype(float)
        if method == "spearman":
            x = x.rank()
            y = y.rank()
        rows.append((date, x.corr(y, method="pearson")))
    return pd.Series(dict(rows)).sort_index().rename("rank_ic")


def forward_return(price_frame: pd.DataFrame, horizon: int) -> pd.DataFrame:
    return price_frame.shift(-horizon).div(price_frame).sub(1.0)


def orient_signal(signal: pd.DataFrame, forward_by_horizon: dict[int, pd.DataFrame], horizon: int = DEFAULT_HORIZON) -> tuple[pd.DataFrame, int, float]:
    raw_ic = cross_sectional_corr_by_date(signal, forward_by_horizon[horizon], method="spearman").mean()
    direction = 1 if raw_ic >= 0 else -1
    return direction * signal, direction, raw_ic


def causal_orient_signal(
    signal: pd.DataFrame,
    forward_by_horizon: dict[int, pd.DataFrame],
    horizon: int = DEFAULT_HORIZON,
    train_window: int = 504,
    min_ic_observations: int = 126,
) -> tuple[pd.DataFrame, int, float, pd.Series, pd.Series]:
    raw_ic_series = cross_sectional_corr_by_date(signal, forward_by_horizon[horizon], method="spearman")
    directions = pd.Series(np.nan, index=signal.index, dtype=float, name="causal_orientation_direction")
    train_mean_ic = pd.Series(np.nan, index=signal.index, dtype=float, name="causal_train_mean_ic")
    for position, date in enumerate(signal.index):
        known_ic = raw_ic_series.iloc[:max(0, position - horizon)].dropna().tail(train_window)
        if len(known_ic) < min_ic_observations:
            continue
        mean_ic = known_ic.mean()
        directions.loc[date] = 1.0 if mean_ic >= 0 else -1.0
        train_mean_ic.loc[date] = mean_ic
    oriented = signal.mul(directions, axis=0)
    latest_direction = int(directions.dropna().iloc[-1]) if directions.notna().any() else 1
    return oriented, latest_direction, raw_ic_series.mean(), directions, train_mean_ic


def ic_summary_for_panel(panel_name: str, signal: pd.DataFrame, forward_by_horizon: dict[int, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for horizon, fwd in forward_by_horizon.items():
        rank_ic = cross_sectional_corr_by_date(signal, fwd, method="spearman")
        rank_ic_vol = rank_ic.std()
        rows.append({
            "panel": panel_name,
            "metric_type": "ic",
            "horizon_days": horizon,
            "mean_rank_ic": rank_ic.mean(),
            "median_rank_ic": rank_ic.median(),
            "rank_ic_vol": rank_ic_vol,
            "rank_icir": rank_ic.mean() / rank_ic_vol if rank_ic_vol and not pd.isna(rank_ic_vol) else np.nan,
            "positive_ic_rate": rank_ic.dropna().gt(0).mean(),
            "observations": rank_ic.notna().sum(),
        })
    return pd.DataFrame(rows)


def bucket_returns(signal: pd.DataFrame, future: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    rows = []
    for date in signal.index.intersection(future.index):
        scores = signal.loc[date].dropna()
        rets = future.loc[date].dropna()
        common = scores.index.intersection(rets.index)
        if len(common) < top_n * 2:
            continue
        scores = scores[common].sort_values(ascending=False)
        rets = rets[common]
        top = scores.head(top_n).index
        bottom = scores.tail(top_n).index
        rows.append({
            "date": date,
            "top_return": rets[top].mean(),
            "bottom_return": rets[bottom].mean(),
            "spread_return": rets[top].mean() - rets[bottom].mean(),
        })
    return pd.DataFrame(rows).set_index("date") if rows else pd.DataFrame(columns=["top_return", "bottom_return", "spread_return"])


def bucket_summary_for_panel(panel_name: str, signal: pd.DataFrame, future: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    buckets = bucket_returns(signal, future, top_n=top_n)
    return pd.DataFrame([{
        "panel": panel_name,
        "metric_type": "bucket",
        "horizon_days": DEFAULT_HORIZON,
        "top_n": top_n,
        "mean_top_return": buckets["top_return"].mean(),
        "mean_bottom_return": buckets["bottom_return"].mean(),
        "mean_top_bottom_spread": buckets["spread_return"].mean(),
        "spread_hit_rate": buckets["spread_return"].dropna().gt(0).mean(),
        "observations": len(buckets),
    }])


def performance_metrics(returns_series: pd.Series, turnover: pd.Series | None = None, periods_per_year: int = 252) -> dict:
    r = returns_series.dropna()
    if r.empty:
        return {"observations": 0, "total_return": np.nan, "cagr": np.nan, "annual_vol": np.nan, "sharpe": np.nan, "sortino": np.nan, "max_drawdown": np.nan, "hit_rate": np.nan, "avg_daily_turnover": np.nan}
    equity = (1 + r).cumprod()
    years = len(r) / periods_per_year
    cagr = equity.iloc[-1] ** (1 / years) - 1 if years > 0 and equity.iloc[-1] > 0 else np.nan
    annual_vol = r.std(ddof=0) * math.sqrt(periods_per_year)
    sharpe = (r.mean() * periods_per_year) / annual_vol if annual_vol and annual_vol > 0 else np.nan
    downside = r[r < 0].std(ddof=0) * math.sqrt(periods_per_year)
    sortino = (r.mean() * periods_per_year) / downside if downside and downside > 0 else np.nan
    drawdown = equity / equity.cummax() - 1
    return {
        "observations": len(r),
        "total_return": equity.iloc[-1] - 1,
        "cagr": cagr,
        "annual_vol": annual_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": drawdown.min(),
        "hit_rate": r.gt(0).mean(),
        "avg_daily_turnover": turnover.reindex(r.index).mean() if turnover is not None else np.nan,
    }


def build_top_signal_weights(
    signal: pd.DataFrame,
    top_n: int = 10,
    min_signal: float = 0.0,
    rebalance_every: int | None = 5,
    rebalance_mask: pd.Series | None = None,
) -> pd.DataFrame:
    weights = pd.DataFrame(0.0, index=signal.index, columns=signal.columns)
    last_weights = pd.Series(0.0, index=signal.columns)
    if rebalance_mask is not None:
        rebalance_mask = rebalance_mask.reindex(signal.index).fillna(False).astype(bool)
    for i, date in enumerate(signal.index):
        should_rebalance = bool(rebalance_mask.loc[date]) if rebalance_mask is not None else (rebalance_every is not None and i % rebalance_every == 0)
        if i == 0:
            should_rebalance = True
        if not should_rebalance:
            weights.loc[date] = last_weights
            continue
        scores = signal.loc[date].dropna().sort_values(ascending=False)
        scores = scores[scores >= min_signal]
        if len(scores) < top_n:
            last_weights = pd.Series(0.0, index=signal.columns)
        else:
            last_weights = pd.Series(0.0, index=signal.columns)
            last_weights.loc[scores.head(top_n).index] = 1.0 / top_n
        weights.loc[date] = last_weights
    return weights


def backtest_weights(weights: pd.DataFrame, next_returns: pd.DataFrame, cost_bps: float) -> dict:
    aligned_returns = next_returns.reindex_like(weights)
    valid_exposure_return = weights.abs().gt(0) & aligned_returns.notna()
    no_exposure = weights.abs().sum(axis=1).eq(0)
    gross = weights.mul(aligned_returns).sum(axis=1, min_count=1).where(valid_exposure_return.any(axis=1) | no_exposure)
    turnover = weights.diff().abs().sum(axis=1, min_count=1).fillna(weights.abs().sum(axis=1))
    costs = turnover * (cost_bps / 10000.0)
    net = gross - costs
    valid_index = net.dropna().index
    return {
        "returns": net.reindex(valid_index),
        "gross_returns": gross.reindex(valid_index),
        "turnover": turnover.reindex(valid_index).fillna(0),
        "costs": costs.reindex(valid_index).fillna(0),
        "weights": weights,
        "metrics": performance_metrics(net, turnover),
    }


def equal_weight_universe_weights(mask: pd.DataFrame) -> pd.DataFrame:
    counts = mask.sum(axis=1).replace(0, np.nan)
    return mask.astype(float).div(counts, axis=0).fillna(0.0)


n500_returns = n500_adj_close.pct_change(fill_method=None)
n500_forward_returns = {h: forward_return(n500_adj_close, h) for h in FORWARD_HORIZONS}
n500_next_returns = n500_adj_close.pct_change(fill_method=None).shift(-1)

n500_history_ok = n500_adj_close.notna().rolling(HIGH_VOL_MIN_HISTORY, min_periods=HIGH_VOL_MIN_HISTORY).sum().shift(1).ge(HIGH_VOL_MIN_HISTORY)
n500_price_ok = n500_close.shift(1).ge(HIGH_VOL_MIN_PRICE)
n500_adv20 = n500_close.mul(n500_volume).rolling(20, min_periods=20).mean().shift(1)
n500_eligible_mask = n500_history_ok & n500_price_ok & n500_adv20.ge(HIGH_VOL_MIN_ADV_RUPEES)

n500_all_raw_alpha = compute_alpha001_inside_universe(n500_adj_close, n500_returns, n500_eligible_mask)
high_vol_raw_alpha = compute_alpha001_inside_universe(n500_adj_close, n500_returns, high_vol_mask_100)

n500_all_alpha, n500_all_direction, n500_all_default_raw_ic, n500_all_causal_direction, n500_all_causal_train_mean_ic = causal_orient_signal(n500_all_raw_alpha, n500_forward_returns)
high_vol_alpha, high_vol_direction, high_vol_default_raw_ic, high_vol_causal_direction, high_vol_causal_train_mean_ic = causal_orient_signal(high_vol_raw_alpha, n500_forward_returns)

n500_all_alpha.to_csv(ARTIFACT_DIR / "nifty500_all_eligible_oriented_alpha001_scores.csv")
high_vol_alpha.to_csv(ARTIFACT_DIR / "high_vol_top100_oriented_alpha001_scores.csv")
n500_all_causal_direction.to_csv(ARTIFACT_DIR / "nifty500_all_eligible_causal_orientation.csv")
high_vol_causal_direction.to_csv(ARTIFACT_DIR / "high_vol_top100_causal_orientation.csv")

panel_metric_parts = []
if "oriented_alpha001" in globals() and "forward_returns" in globals():
    panel_metric_parts.append(ic_summary_for_panel("nifty50_current", oriented_alpha001, forward_returns))
    panel_metric_parts.append(bucket_summary_for_panel("nifty50_current", oriented_alpha001, forward_returns[DEFAULT_HORIZON], top_n=10))
panel_metric_parts.append(ic_summary_for_panel("nifty500_all_eligible", n500_all_alpha, n500_forward_returns))
panel_metric_parts.append(bucket_summary_for_panel("nifty500_all_eligible", n500_all_alpha, n500_forward_returns[DEFAULT_HORIZON], top_n=10))
panel_metric_parts.append(ic_summary_for_panel("nifty500_high_vol_top100", high_vol_alpha, n500_forward_returns))
panel_metric_parts.append(bucket_summary_for_panel("nifty500_high_vol_top100", high_vol_alpha, n500_forward_returns[DEFAULT_HORIZON], top_n=10))

high_vol_primary_top_n = 10
high_vol_primary_min_signal = 0.0
high_vol_primary_rebalance = "weekly_reconstitution"
high_vol_primary_rebalance_mask = weekly_reconstitution_mask(high_vol_alpha.index)
high_vol_primary_weights = build_top_signal_weights(high_vol_alpha, top_n=high_vol_primary_top_n, min_signal=high_vol_primary_min_signal, rebalance_mask=high_vol_primary_rebalance_mask)
high_vol_benchmark_weights = equal_weight_universe_weights(high_vol_mask_100)

cost_rows = []
for cost_bps in [10.0, 20.0, 35.0]:
    alpha_bt = backtest_weights(high_vol_primary_weights, n500_next_returns, cost_bps=cost_bps)
    benchmark_bt = backtest_weights(high_vol_benchmark_weights, n500_next_returns, cost_bps=cost_bps)
    for label, bt in [("alpha_top_bucket", alpha_bt), ("equal_weight_high_vol_universe", benchmark_bt)]:
        row = {"panel": "nifty500_high_vol_top100", "metric_type": "strategy_cost_sensitivity", "strategy": label, "cost_bps": cost_bps}
        row.update(bt["metrics"])
        cost_rows.append(row)
    excess = alpha_bt["returns"].sub(benchmark_bt["returns"], fill_value=0.0)
    row = {"panel": "nifty500_high_vol_top100", "metric_type": "strategy_cost_sensitivity", "strategy": "alpha_minus_high_vol_equal_weight", "cost_bps": cost_bps}
    row.update(performance_metrics(excess))
    cost_rows.append(row)
panel_metric_parts.append(pd.DataFrame(cost_rows))

basket_rows = []
for basket_size, buffer_size in [(50, 75), (100, 125), (150, 175)]:
    mask, report = build_dynamic_high_vol_universe(
        n500_adj_close,
        n500_close,
        n500_volume,
        basket_size=basket_size,
        buffer_size=buffer_size,
        vol_lookback=HIGH_VOL_LOOKBACK,
        min_history=HIGH_VOL_MIN_HISTORY,
        min_price=HIGH_VOL_MIN_PRICE,
        min_adv_rupees=HIGH_VOL_MIN_ADV_RUPEES,
    )
    raw_signal = compute_alpha001_inside_universe(n500_adj_close, n500_returns, mask)
    signal, direction, raw_ic, _, _ = causal_orient_signal(raw_signal, n500_forward_returns)
    weights = build_top_signal_weights(signal, top_n=min(10, max(5, basket_size // 10)), min_signal=0.0, rebalance_mask=weekly_reconstitution_mask(signal.index))
    bt = backtest_weights(weights, n500_next_returns, cost_bps=20.0)
    buckets = bucket_returns(signal, n500_forward_returns[DEFAULT_HORIZON], top_n=min(10, max(5, basket_size // 10)))
    row = {
        "panel": f"nifty500_high_vol_top{basket_size}",
        "metric_type": "basket_size_sensitivity",
        "basket_size": basket_size,
        "buffer_size": buffer_size,
        "orientation": "raw" if direction == 1 else "inverted",
        "default_raw_mean_ic": raw_ic,
        "mean_5d_rank_ic": cross_sectional_corr_by_date(signal, n500_forward_returns[DEFAULT_HORIZON]).mean(),
        "mean_top_bottom_spread": buckets["spread_return"].mean(),
        "average_weekly_membership_turnover": report["membership_turnover_fraction"].mean() if not report.empty else np.nan,
    }
    row.update(bt["metrics"])
    basket_rows.append(row)
panel_metric_parts.append(pd.DataFrame(basket_rows))

high_vol_alpha001_metrics = pd.concat(panel_metric_parts, ignore_index=True, sort=False)
high_vol_alpha001_metrics.to_csv(ARTIFACT_DIR / "high_vol_alpha001_metrics.csv", index=False)

print("Panel comparison and high-volatility strategy metrics")
display(high_vol_alpha001_metrics)

fig, ax = plt.subplots(figsize=(12, 5))
primary_bt_20 = backtest_weights(high_vol_primary_weights, n500_next_returns, cost_bps=20.0)
benchmark_bt_20 = backtest_weights(high_vol_benchmark_weights, n500_next_returns, cost_bps=20.0)
(1 + primary_bt_20["returns"].fillna(0)).cumprod().plot(ax=ax, label="Alpha#1 top bucket, 20 bps")
(1 + benchmark_bt_20["returns"].fillna(0)).cumprod().plot(ax=ax, label="Equal-weight high-vol universe, 20 bps")
ax.set_title("Dynamic high-vol universe: Alpha#1 vs equal-weight benchmark")
ax.set_ylabel("Growth of 1")
ax.legend()
plt.show()

## 16. Causal Regime Gates for High-Volatility Alpha#1

The goal here is not to label regimes after the fact. The gates use observable market information available at the signal timestamp, and rolling historical thresholds rather than full-sample cutoffs.

We test two ideas:

1. **Level regimes**: high/medium/low volatility, dispersion, breadth, trend, and momentum.
2. **Early warning transitions**: volatility rising, dispersion expanding, breadth weakening, and momentum breaking.

For each gate, the report compares Alpha#1-selected high-vol returns against the equal-weight high-vol universe in the same active periods. That tells us whether Alpha#1 adds information beyond simply being in a high-volatility basket.

In [ ]:
def rolling_tercile_label(series: pd.Series, low_label: str, mid_label: str, high_label: str, window: int = 252, min_periods: int = 126) -> pd.Series:
    clean = series.replace([np.inf, -np.inf], np.nan)
    history = clean.shift(1)
    low_cut = history.rolling(window, min_periods=min_periods).quantile(0.33)
    high_cut = history.rolling(window, min_periods=min_periods).quantile(0.67)
    labels = pd.Series("unknown", index=series.index, dtype="object")
    labels.loc[clean <= low_cut] = low_label
    labels.loc[(clean > low_cut) & (clean < high_cut)] = mid_label
    labels.loc[clean >= high_cut] = high_label
    return labels


def causal_regime_features(index_close: pd.Series, universe_returns: pd.DataFrame, universe_volume: pd.DataFrame) -> pd.DataFrame:
    idx = index_close.dropna().copy()
    idx_ret = idx.pct_change(fill_method=None)
    idx_vol20 = idx_ret.rolling(20, min_periods=20).std() * math.sqrt(ANNUALIZATION)
    idx_vol_change5 = idx_vol20.pct_change(5, fill_method=None)
    idx_mom20 = idx.pct_change(20, fill_method=None)
    idx_mom63 = idx.pct_change(63, fill_method=None)
    sma50 = idx.rolling(50, min_periods=50).mean()
    sma200 = idx.rolling(200, min_periods=200).mean()

    breadth20 = universe_returns.gt(0).mean(axis=1).rolling(20, min_periods=20).mean()
    dispersion20 = universe_returns.std(axis=1).rolling(20, min_periods=20).mean() * math.sqrt(ANNUALIZATION)
    volume_intensity = universe_volume.div(universe_volume.rolling(20, min_periods=20).mean()).replace([np.inf, -np.inf], np.nan).median(axis=1).rolling(5, min_periods=5).mean()

    frame = pd.DataFrame(index=idx.index)
    frame["market_vol_level"] = rolling_tercile_label(idx_vol20, "low", "middle", "high")
    frame["market_vol_change"] = rolling_tercile_label(idx_vol_change5, "falling", "stable", "rising")
    frame["market_mom20"] = rolling_tercile_label(idx_mom20, "weak", "middle", "strong")
    frame["market_mom63"] = rolling_tercile_label(idx_mom63, "weak", "middle", "strong")
    frame["breadth_level"] = rolling_tercile_label(breadth20, "weak", "middle", "broad")
    frame["dispersion_level"] = rolling_tercile_label(dispersion20, "low", "middle", "high")
    frame["volume_intensity"] = rolling_tercile_label(volume_intensity, "quiet", "normal", "heavy")
    frame["trend_state"] = "sideways"
    frame.loc[(idx > sma50) & (sma50 > sma200), "trend_state"] = "bull"
    frame.loc[(idx < sma50) & (sma50 < sma200), "trend_state"] = "bear"
    frame.loc[sma50.isna() | sma200.isna(), "trend_state"] = "unknown"
    frame["early_warning_stress"] = (
        frame["market_vol_change"].eq("rising")
        | frame["dispersion_level"].eq("high")
        | frame["breadth_level"].eq("weak")
        | frame["market_mom63"].eq("weak")
    )
    frame["stress_without_late_trend_filter"] = (
        frame["market_vol_change"].eq("rising")
        | frame["dispersion_level"].eq("high")
        | frame["breadth_level"].eq("weak")
    )
    return frame


def metrics_for_gate(name: str, gate: pd.Series, alpha_weights: pd.DataFrame, benchmark_weights: pd.DataFrame, next_returns: pd.DataFrame, cost_bps: float = 20.0) -> dict:
    aligned_gate = gate.reindex(alpha_weights.index).fillna(False).astype(bool)
    gated_alpha_weights = alpha_weights.where(aligned_gate, 0.0)
    gated_benchmark_weights = benchmark_weights.where(aligned_gate, 0.0)
    alpha_bt = backtest_weights(gated_alpha_weights, next_returns, cost_bps=cost_bps)
    benchmark_bt = backtest_weights(gated_benchmark_weights, next_returns, cost_bps=cost_bps)
    excess_returns = alpha_bt["returns"].sub(benchmark_bt["returns"], fill_value=0.0)
    excess_metrics = performance_metrics(excess_returns)
    row = {
        "gate": name,
        "active_days": int(aligned_gate.sum()),
        "active_fraction": float(aligned_gate.mean()),
    }
    for prefix, metrics in [
        ("alpha", alpha_bt["metrics"]),
        ("benchmark", benchmark_bt["metrics"]),
        ("excess", excess_metrics),
    ]:
        for key, value in metrics.items():
            row[f"{prefix}_{key}"] = value
    return row


def conditional_alpha_information_report(condition_frame: pd.DataFrame, alpha_signal: pd.DataFrame, forward: pd.DataFrame, spread: pd.Series, alpha_returns: pd.Series, benchmark_returns: pd.Series) -> pd.DataFrame:
    rows = []
    rank_ic = cross_sectional_corr_by_date(alpha_signal, forward, method="spearman")
    excess = alpha_returns.sub(benchmark_returns, fill_value=0.0)
    for condition in condition_frame.columns:
        labels = condition_frame[condition]
        if labels.dtype == bool:
            label_items = {"true": labels, "false": ~labels}
        else:
            label_items = {str(label): labels.eq(label) for label in sorted(labels.dropna().unique())}
        for label, mask in label_items.items():
            mask = mask.reindex(rank_ic.index).fillna(False).astype(bool)
            if mask.sum() < 60:
                continue
            perf = performance_metrics(alpha_returns.where(mask))
            bench = performance_metrics(benchmark_returns.where(mask))
            excess_perf = performance_metrics(excess.where(mask))
            rows.append({
                "condition": condition,
                "bucket": label,
                "days": int(mask.sum()),
                "mean_rank_ic": rank_ic.where(mask).mean(),
                "positive_ic_rate": rank_ic.where(mask).dropna().gt(0).mean(),
                "mean_top_bottom_spread": spread.where(mask).mean(),
                "alpha_sharpe": perf["sharpe"],
                "benchmark_sharpe": bench["sharpe"],
                "excess_sharpe": excess_perf["sharpe"],
                "excess_total_return": excess_perf["total_return"],
            })
    return pd.DataFrame(rows)


high_vol_regime_features = causal_regime_features(nifty_close, n500_returns, n500_volume)
high_vol_spread = bucket_returns(high_vol_alpha, n500_forward_returns[DEFAULT_HORIZON], top_n=10)["spread_return"]
primary_bt_20 = backtest_weights(high_vol_primary_weights, n500_next_returns, cost_bps=20.0)
benchmark_bt_20 = backtest_weights(high_vol_benchmark_weights, n500_next_returns, cost_bps=20.0)

level_gate_rows = conditional_alpha_information_report(
    high_vol_regime_features,
    high_vol_alpha,
    n500_forward_returns[DEFAULT_HORIZON],
    high_vol_spread,
    primary_bt_20["returns"],
    benchmark_bt_20["returns"],
)

candidate_gates = {
    "ungated": pd.Series(True, index=high_vol_alpha.index),
    "vol_level_high": high_vol_regime_features["market_vol_level"].eq("high"),
    "vol_change_rising": high_vol_regime_features["market_vol_change"].eq("rising"),
    "dispersion_high": high_vol_regime_features["dispersion_level"].eq("high"),
    "breadth_weak": high_vol_regime_features["breadth_level"].eq("weak"),
    "market_mom63_weak": high_vol_regime_features["market_mom63"].eq("weak"),
    "early_warning_stress": high_vol_regime_features["early_warning_stress"],
    "stress_without_trend": high_vol_regime_features["stress_without_late_trend_filter"],
}

gated_strategy_rows = [
    metrics_for_gate(name, gate, high_vol_primary_weights, high_vol_benchmark_weights, n500_next_returns, cost_bps=20.0)
    for name, gate in candidate_gates.items()
]
gated_strategy_report = pd.DataFrame(gated_strategy_rows)
gated_strategy_report.insert(0, "report_type", "gated_strategy")
level_gate_rows.insert(0, "report_type", "conditional_information")

high_vol_regime_gate_report = pd.concat([gated_strategy_report, level_gate_rows], ignore_index=True, sort=False)
high_vol_regime_gate_report.to_csv(ARTIFACT_DIR / "high_vol_regime_gate_report.csv", index=False)
high_vol_regime_features.to_csv(ARTIFACT_DIR / "high_vol_causal_regime_features.csv")

print("Gated high-vol Alpha#1 strategy comparison")
display(gated_strategy_report.sort_values("excess_sharpe", ascending=False))
print("Conditional information pockets")
display(level_gate_rows.sort_values("mean_rank_ic", ascending=False).head(20))

## 17. Latest High-Volatility Alpha#1 Candidates

This is the current candidate table from the dynamic high-volatility basket. It is not a trade ticket. It shows which names are active members of the high-volatility universe and rank in the top Alpha#1 bucket under the current primary rule.

In [ ]:
latest_hv_date = high_vol_alpha.dropna(how="all").index[-1]
latest_scores = high_vol_alpha.loc[latest_hv_date].dropna().sort_values(ascending=False)
latest_selected = latest_scores.head(high_vol_primary_top_n)
latest_vol20 = n500_returns.rolling(HIGH_VOL_LOOKBACK, min_periods=HIGH_VOL_LOOKBACK).std().shift(1) * math.sqrt(ANNUALIZATION)
latest_adv20 = n500_close.mul(n500_volume).rolling(20, min_periods=20).mean().shift(1)
latest_regime_row = high_vol_regime_features.reindex(high_vol_alpha.index).loc[latest_hv_date]

candidate_rows = []
for symbol, score in latest_selected.items():
    candidate_rows.append({
        "date": latest_hv_date,
        "symbol": symbol,
        "alpha": "alpha001_high_vol_top100",
        "suggested_direction": "long",
        "target_weight_equal_weight": 1.0 / len(latest_selected) if len(latest_selected) else np.nan,
        "oriented_signal_score": score,
        "raw_alpha001_score": high_vol_raw_alpha.loc[latest_hv_date, symbol] if symbol in high_vol_raw_alpha.columns else np.nan,
        "signal_orientation": "raw" if high_vol_direction == 1 else "inverted",
        "active_high_vol_member": bool(high_vol_mask_100.loc[latest_hv_date, symbol]),
        "lagged_20d_vol_ann": latest_vol20.loc[latest_hv_date, symbol] if symbol in latest_vol20.columns else np.nan,
        "lagged_20d_adv_rupees": latest_adv20.loc[latest_hv_date, symbol] if symbol in latest_adv20.columns else np.nan,
        "market_vol_level": latest_regime_row.get("market_vol_level"),
        "market_vol_change": latest_regime_row.get("market_vol_change"),
        "breadth_level": latest_regime_row.get("breadth_level"),
        "dispersion_level": latest_regime_row.get("dispersion_level"),
        "market_mom63": latest_regime_row.get("market_mom63"),
        "early_warning_stress": latest_regime_row.get("early_warning_stress"),
        "reason": f"Top {high_vol_primary_top_n} Alpha#1 score inside weekly top-{HIGH_VOL_BASKET_SIZE} lagged-volatility basket",
    })

high_vol_latest_candidates = pd.DataFrame(candidate_rows)
high_vol_latest_candidates.to_csv(ARTIFACT_DIR / "high_vol_latest_candidates.csv", index=False)

sanity_rows = []
alpha_score_values = high_vol_alpha.stack().dropna()
alpha_bounds_ok = alpha_score_values.between(-0.5, 0.5).all()
weekly_reconstitution_dates = pd.DatetimeIndex(pd.to_datetime(high_vol_universe_report["date"])) if not high_vol_universe_report.empty else pd.DatetimeIndex([])
membership_change_count = high_vol_mask_100.astype(int).diff().abs().sum(axis=1).fillna(0)
non_reconstitution_change_dates = membership_change_count[membership_change_count.gt(0) & ~membership_change_count.index.isin(weekly_reconstitution_dates)]
history_counts = n500_adj_close.notna().rolling(HIGH_VOL_MIN_HISTORY, min_periods=HIGH_VOL_MIN_HISTORY).sum().shift(1)
selection_mask = high_vol_primary_weights.ne(0)
rebalance_selection_mask = selection_mask.copy()
rebalance_selection_mask.loc[~high_vol_primary_rebalance_mask.reindex(selection_mask.index).fillna(False)] = False
forward_available_dates = n500_forward_returns[DEFAULT_HORIZON].notna().any(axis=1).reindex(rebalance_selection_mask.index).fillna(False)
evaluable_selection_mask = rebalance_selection_mask.copy()
evaluable_selection_mask.loc[~forward_available_dates] = False
expected_selected_points = int(evaluable_selection_mask.sum().sum())
selected_forward_points = int(n500_forward_returns[DEFAULT_HORIZON].where(evaluable_selection_mask).stack().dropna().shape[0])
selected_history_values = history_counts.where(evaluable_selection_mask).stack().dropna()
selected_data_ok = bool(expected_selected_points > 0 and selected_forward_points == expected_selected_points and len(selected_history_values) == expected_selected_points and selected_history_values.ge(HIGH_VOL_MIN_HISTORY).all())
sanity_rows.append({"check": "alpha001_scores_bounded", "passed": bool(alpha_bounds_ok), "detail": "finite scores should remain inside [-0.5, 0.5]"})
sanity_rows.append({"check": "latest_candidates_are_active_members", "passed": bool(high_vol_latest_candidates["active_high_vol_member"].all()) if not high_vol_latest_candidates.empty else False, "detail": "selected names must be active high-vol members"})
sanity_rows.append({"check": "median_dynamic_basket_size_near_target", "passed": bool(universe_daily_count[universe_daily_count > 0].median() >= HIGH_VOL_BASKET_SIZE * 0.9), "detail": f"target basket size {HIGH_VOL_BASKET_SIZE}"})
sanity_rows.append({"check": "membership_changes_only_on_reconstitution_dates", "passed": bool(non_reconstitution_change_dates.empty), "detail": f"non-reconstitution change dates: {len(non_reconstitution_change_dates)}"})
sanity_rows.append({"check": "selected_names_have_history_and_forward_returns", "passed": selected_data_ok, "detail": f"evaluable rebalance selections: {expected_selected_points}"})
high_vol_sanity_checks = pd.DataFrame(sanity_rows)
high_vol_sanity_checks.to_csv(ARTIFACT_DIR / "high_vol_sanity_checks.csv", index=False)

print("Latest high-volatility Alpha#1 candidate table")
display(high_vol_latest_candidates)
print("High-volatility extension sanity checks")
display(high_vol_sanity_checks)

## 18. Index-Guided Volatility Baskets

The stock-level high-volatility basket is useful, but market practitioners also watch volatility-heavy index neighborhoods: High Beta, Midcap, Smallcap, Bank, and IT. This section loads current official Nifty constituent files for those baskets, recomputes Alpha#1 ranks inside each basket, and compares their IC, bucket spread, turnover, and 20 bps strategy performance. These are current-constituent research baskets, so treat the historical results as diagnostics rather than a fully historical index-membership backtest.

In [ ]:

VOLATILE_INDEX_DIR = NIFTY500_DATA_DIR / "volatile_index_constituents"
VOLATILE_INDEX_DIR.mkdir(parents=True, exist_ok=True)
VOLATILE_INDEX_REFRESH = False
VOLATILE_INDEX_METRICS_PATH = ARTIFACT_DIR / "volatile_index_alpha001_metrics.csv"
VOLATILE_INDEX_CONSTITUENT_REPORT_PATH = ARTIFACT_DIR / "volatile_index_constituent_report.csv"
VOLATILE_INDEX_LATEST_CANDIDATES_PATH = ARTIFACT_DIR / "volatile_index_latest_candidates.csv"

VOLATILE_INDEX_SPECS = {
    "nifty_high_beta_50": {
        "name": "NIFTY High Beta 50",
        "urls": [
            "https://www.niftyindices.com/IndexConstituent/nifty_high_beta50_index.csv",
            "https://archives.nseindia.com/content/indices/nifty_high_beta50_index.csv",
        ],
    },
    "nifty_midcap_50": {
        "name": "NIFTY Midcap 50",
        "urls": [
            "https://www.niftyindices.com/IndexConstituent/ind_niftymidcap50list.csv",
            "https://archives.nseindia.com/content/indices/ind_niftymidcap50list.csv",
        ],
    },
    "nifty_midcap_150": {
        "name": "NIFTY Midcap 150",
        "urls": [
            "https://www.niftyindices.com/IndexConstituent/ind_niftymidcap150list.csv",
            "https://archives.nseindia.com/content/indices/ind_niftymidcap150list.csv",
        ],
    },
    "nifty_smallcap_250": {
        "name": "NIFTY Smallcap 250",
        "urls": [
            "https://www.niftyindices.com/IndexConstituent/ind_niftysmallcap250list.csv",
            "https://archives.nseindia.com/content/indices/ind_niftysmallcap250list.csv",
        ],
    },
    "nifty_it": {
        "name": "NIFTY IT",
        "urls": [
            "https://www.niftyindices.com/IndexConstituent/ind_niftyitlist.csv",
            "https://archives.nseindia.com/content/indices/ind_niftyitlist.csv",
        ],
    },
    "nifty_bank": {
        "name": "NIFTY Bank",
        "urls": [
            "https://www.niftyindices.com/IndexConstituent/ind_niftybanklist.csv",
            "https://archives.nseindia.com/content/indices/ind_niftybanklist.csv",
        ],
    },
}


def load_volatile_index_constituents(slug: str, spec: dict, refresh: bool = False) -> pd.DataFrame:
    path = VOLATILE_INDEX_DIR / f"{slug}.csv"
    if path.exists() and not refresh:
        frame = pd.read_csv(path)
        print(f"Loaded cached {spec['name']}: {len(frame)} symbols")
        return frame

    headers = {
        "User-Agent": "Mozilla/5.0 Alpha001 research notebook",
        "Accept": "text/csv,application/csv,text/plain,*/*",
        "Referer": "https://www.niftyindices.com/",
    }
    errors = []
    for url in spec["urls"]:
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            if "Symbol" not in response.text[:1000]:
                raise ValueError("response did not look like a constituent CSV")
            frame = parse_constituent_csv(response.text, url)
            frame.insert(0, "index_slug", slug)
            frame.insert(1, "index_name", spec["name"])
            frame.to_csv(path, index=False)
            print(f"Downloaded {spec['name']} from {url}: {len(frame)} symbols")
            return frame
        except Exception as exc:
            errors.append({"index_slug": slug, "index_name": spec["name"], "url": url, "error": repr(exc)})
    pd.DataFrame(errors).to_csv(VOLATILE_INDEX_DIR / f"{slug}_fetch_errors.csv", index=False)
    raise RuntimeError(f"Could not fetch {spec['name']} and no cache exists. See {VOLATILE_INDEX_DIR / f'{slug}_fetch_errors.csv'}")


volatile_index_frames = {}
volatile_index_errors = []
for slug, spec in VOLATILE_INDEX_SPECS.items():
    try:
        volatile_index_frames[slug] = load_volatile_index_constituents(slug, spec, refresh=VOLATILE_INDEX_REFRESH)
    except Exception as exc:
        volatile_index_errors.append({"index_slug": slug, "index_name": spec["name"], "error": repr(exc)})

if volatile_index_errors:
    pd.DataFrame(volatile_index_errors).to_csv(ARTIFACT_DIR / "volatile_index_fetch_errors.csv", index=False)
    display(pd.DataFrame(volatile_index_errors))


def static_index_mask(symbols: list[str], eligible_mask: pd.DataFrame) -> pd.DataFrame:
    available = [symbol for symbol in symbols if symbol in eligible_mask.columns]
    mask = pd.DataFrame(False, index=eligible_mask.index, columns=eligible_mask.columns)
    if available:
        mask.loc[:, available] = True
    return mask & eligible_mask.reindex_like(mask).fillna(False)


def ew_returns_for_mask(mask: pd.DataFrame, returns_frame: pd.DataFrame) -> pd.Series:
    weights = equal_weight_universe_weights(mask)
    return weights.mul(returns_frame.reindex_like(weights)).sum(axis=1).rename("equal_weight_return")


volatile_index_masks = {}
constituent_report_rows = []
metric_parts = []
latest_candidate_rows = []

for slug, frame in volatile_index_frames.items():
    index_name = VOLATILE_INDEX_SPECS[slug]["name"]
    symbols = frame["Symbol"].dropna().astype(str).str.upper().drop_duplicates().tolist()
    available_symbols = [symbol for symbol in symbols if symbol in n500_adj_close.columns]
    missing_symbols = sorted(set(symbols) - set(available_symbols))
    active_mask = static_index_mask(symbols, n500_eligible_mask)
    volatile_index_masks[slug] = active_mask
    latest_active_count = int(active_mask.iloc[-1].sum()) if len(active_mask) else 0
    member_count = len(available_symbols)
    top_n = min(10, max(3, int(round(member_count * 0.10))))
    min_names_for_ic = min(10, max(5, member_count // 2))

    constituent_report_rows.append({
        "index_slug": slug,
        "index_name": index_name,
        "constituents": len(symbols),
        "available_in_nifty500_cache": len(available_symbols),
        "latest_eligible_count": latest_active_count,
        "strategy_top_n": top_n,
        "missing_symbols": ",".join(missing_symbols),
        "source_url": frame["source_url"].iloc[0] if "source_url" in frame.columns and len(frame) else "",
    })

    if member_count < 6:
        continue

    raw_signal = compute_alpha001_inside_universe(n500_adj_close, n500_returns, active_mask)
    signal, direction, raw_ic, _, _ = causal_orient_signal(raw_signal, n500_forward_returns)
    latest_date = signal.dropna(how="all").index.max()

    for horizon, fwd in n500_forward_returns.items():
        rank_ic = cross_sectional_corr_by_date(signal, fwd, method="spearman", min_names=min_names_for_ic)
        metric_parts.append(pd.DataFrame([{
            "panel": slug,
            "index_name": index_name,
            "metric_type": "ic",
            "horizon_days": horizon,
            "orientation": "raw" if direction == 1 else "inverted",
            "default_raw_mean_ic": raw_ic,
            "mean_rank_ic": rank_ic.mean(),
            "median_rank_ic": rank_ic.median(),
            "rank_ic_vol": rank_ic.std(),
            "rank_icir": rank_ic.mean() / rank_ic.std() if rank_ic.std() and not pd.isna(rank_ic.std()) else np.nan,
            "positive_ic_rate": rank_ic.dropna().gt(0).mean(),
            "observations": rank_ic.notna().sum(),
            "min_names_for_ic": min_names_for_ic,
        }]))

    buckets = bucket_returns(signal, n500_forward_returns[DEFAULT_HORIZON], top_n=top_n)
    metric_parts.append(pd.DataFrame([{
        "panel": slug,
        "index_name": index_name,
        "metric_type": "bucket",
        "horizon_days": DEFAULT_HORIZON,
        "top_n": top_n,
        "mean_top_return": buckets["top_return"].mean(),
        "mean_bottom_return": buckets["bottom_return"].mean(),
        "mean_top_bottom_spread": buckets["spread_return"].mean(),
        "spread_hit_rate": buckets["spread_return"].gt(0).mean(),
        "observations": len(buckets),
    }]))

    strategy_weights = build_top_signal_weights(signal, top_n=top_n, min_signal=0.0, rebalance_mask=weekly_reconstitution_mask(signal.index))
    benchmark_weights = equal_weight_universe_weights(active_mask)
    alpha_bt = backtest_weights(strategy_weights, n500_next_returns, cost_bps=20.0)
    benchmark_bt = backtest_weights(benchmark_weights, n500_next_returns, cost_bps=20.0)
    excess_returns = alpha_bt["returns"].sub(benchmark_bt["returns"], fill_value=0.0)
    for strategy_name, metrics in [
        ("alpha_top_bucket", alpha_bt["metrics"]),
        ("equal_weight_index_basket", benchmark_bt["metrics"]),
        ("alpha_minus_equal_weight", performance_metrics(excess_returns)),
    ]:
        row = {
            "panel": slug,
            "index_name": index_name,
            "metric_type": "strategy_20bps",
            "strategy": strategy_name,
            "top_n": top_n,
            "cost_bps": 20.0,
            "avg_active_names": active_mask.sum(axis=1).replace(0, np.nan).mean(),
        }
        row.update(metrics)
        metric_parts.append(pd.DataFrame([row]))

    if pd.notna(latest_date):
        latest_scores = signal.loc[latest_date].dropna().sort_values(ascending=False).head(top_n)
        for symbol, score in latest_scores.items():
            latest_candidate_rows.append({
                "date": latest_date,
                "index_slug": slug,
                "index_name": index_name,
                "symbol": symbol,
                "target_weight_equal_weight": 1.0 / len(latest_scores) if len(latest_scores) else np.nan,
                "oriented_signal_score": score,
                "signal_orientation": "raw" if direction == 1 else "inverted",
                "active_member": bool(active_mask.loc[latest_date, symbol]) if symbol in active_mask.columns else False,
            })

volatile_index_constituent_report = pd.DataFrame(constituent_report_rows)
volatile_index_constituent_report.to_csv(VOLATILE_INDEX_CONSTITUENT_REPORT_PATH, index=False)
volatile_index_alpha001_metrics = pd.concat(metric_parts, ignore_index=True, sort=False) if metric_parts else pd.DataFrame()
volatile_index_alpha001_metrics.to_csv(VOLATILE_INDEX_METRICS_PATH, index=False)
volatile_index_latest_candidates = pd.DataFrame(latest_candidate_rows)
volatile_index_latest_candidates.to_csv(VOLATILE_INDEX_LATEST_CANDIDATES_PATH, index=False)

print("Volatile index constituent coverage")
display(volatile_index_constituent_report)
print("Volatile index Alpha#1 strategy comparison, 20 bps")
strategy_view = volatile_index_alpha001_metrics.query("metric_type == 'strategy_20bps'") if not volatile_index_alpha001_metrics.empty else pd.DataFrame()
display(strategy_view.sort_values(["panel", "strategy"]))
print("Latest candidates by volatile index basket")
display(volatile_index_latest_candidates)


## 19. Expanded High-Volatility Parent Universe

The previous dynamic basket used NIFTY 500 as the parent. This section expands the parent with official NIFTY Total Market and Microcap 250 constituents, plus the volatility-aware feeder indices loaded above. Only missing symbols are downloaded; existing NIFTY 500 OHLCV caches are reused. The alpha math stays unchanged: eligibility is decided first with lagged data, then Alpha#1 ranks are recomputed only inside the active expanded high-volatility basket.

In [ ]:

EXPANDED_PARENT_DIR = Path("research/data/expanded_high_vol_parent")
EXPANDED_PARENT_DIR.mkdir(parents=True, exist_ok=True)
EXPANDED_PARENT_CONSTITUENT_PATH = EXPANDED_PARENT_DIR / "expanded_parent_constituents.csv"
EXPANDED_DOWNLOAD_REPORT_PATH = ARTIFACT_DIR / "expanded_high_vol_download_report.csv"
EXPANDED_UNIVERSE_REPORT_PATH = ARTIFACT_DIR / "expanded_high_vol_universe_report.csv"
EXPANDED_UNIVERSE_MASK_PATH = ARTIFACT_DIR / "expanded_high_vol_universe_mask_top100.csv"
EXPANDED_ALPHA_METRICS_PATH = ARTIFACT_DIR / "expanded_high_vol_alpha001_metrics.csv"
EXPANDED_LATEST_CANDIDATES_PATH = ARTIFACT_DIR / "expanded_high_vol_latest_candidates.csv"
EXPANDED_PARENT_REFRESH = False
EXPANDED_PARENT_PRICE_REFRESH = False
EXPANDED_DOWNLOAD_CHUNK_SIZE = 75

EXPANDED_PARENT_EXTRA_SPECS = {
    "nifty_total_market": {
        "name": "NIFTY Total Market",
        "urls": [
            "https://www.niftyindices.com/IndexConstituent/ind_niftytotalmarket_list.csv",
            "https://archives.nseindia.com/content/indices/ind_niftytotalmarket_list.csv",
        ],
    },
    "nifty_microcap_250": {
        "name": "NIFTY Microcap 250",
        "urls": [
            "https://www.niftyindices.com/IndexConstituent/ind_niftymicrocap250_list.csv",
            "https://archives.nseindia.com/content/indices/ind_niftymicrocap250_list.csv",
        ],
    },
}

EXPANDED_FIELD_FILES = {
    "open": EXPANDED_PARENT_DIR / "open.csv",
    "high": EXPANDED_PARENT_DIR / "high.csv",
    "low": EXPANDED_PARENT_DIR / "low.csv",
    "close": EXPANDED_PARENT_DIR / "close.csv",
    "adj_close": EXPANDED_PARENT_DIR / "adj_close.csv",
    "volume": EXPANDED_PARENT_DIR / "volume.csv",
}

expanded_extra_frames = {}
for slug, spec in EXPANDED_PARENT_EXTRA_SPECS.items():
    try:
        expanded_extra_frames[slug] = load_volatile_index_constituents(slug, spec, refresh=EXPANDED_PARENT_REFRESH)
    except Exception as exc:
        print(f"Could not load {spec['name']}: {exc!r}")

source_frames = []
if "nifty500_constituents" in globals():
    base = nifty500_constituents.copy()
    base["index_slug"] = "nifty500"
    base["index_name"] = "NIFTY 500"
    source_frames.append(base)
for slug, frame in {**volatile_index_frames, **expanded_extra_frames}.items():
    source_frames.append(frame.copy())

expanded_source = pd.concat(source_frames, ignore_index=True, sort=False)
expanded_source["Symbol"] = expanded_source["Symbol"].astype(str).str.strip().str.upper()
expanded_source = expanded_source[expanded_source["Symbol"].ne("") & expanded_source["Symbol"].ne("NAN")]
source_map = (
    expanded_source.groupby("Symbol")
    .agg(
        company_name=("Company Name", "first"),
        industry=("Industry", "first"),
        source_indices=("index_name", lambda values: ",".join(sorted(set(map(str, values))))),
        source_slugs=("index_slug", lambda values: ",".join(sorted(set(map(str, values))))),
    )
    .reset_index()
    .sort_values("Symbol")
)
source_map["YahooTicker"] = source_map["Symbol"].map(lambda symbol: f"{symbol}.NS")
source_map.to_csv(EXPANDED_PARENT_CONSTITUENT_PATH, index=False)
expanded_symbols = source_map["Symbol"].tolist()
print(f"Expanded parent symbols: {len(expanded_symbols)}")
print(f"NIFTY500 symbols in cache: {n500_adj_close.shape[1]}")
print(f"New parent symbols beyond current NIFTY500 cache: {len(set(expanded_symbols) - set(n500_adj_close.columns))}")


def download_expanded_parent_ohlcv(symbols: list[str], refresh: bool = False, chunk_size: int = 75) -> dict[str, pd.DataFrame]:
    cache_ready = all(path.exists() for path in EXPANDED_FIELD_FILES.values())
    if cache_ready and not refresh:
        data = {name: read_frame(path) for name, path in EXPANDED_FIELD_FILES.items()}
        print(f"Loaded cached expanded parent OHLCV from {EXPANDED_PARENT_DIR}")
        return data

    existing_data = {
        "open": n500_open,
        "high": n500_high,
        "low": n500_low,
        "close": n500_close,
        "adj_close": n500_adj_close,
        "volume": n500_volume,
    }
    existing_symbols = set(n500_adj_close.columns)
    missing_symbols = [symbol for symbol in symbols if symbol not in existing_symbols]
    field_parts = {field: [existing_data[field].reindex(columns=[s for s in symbols if s in existing_symbols])] for field in EXPANDED_FIELD_FILES}
    report_rows = []

    for symbol in symbols:
        if symbol in existing_symbols:
            close_rows = int(n500_close[symbol].notna().sum()) if symbol in n500_close else 0
            report_rows.append({"symbol": symbol, "ticker": f"{symbol}.NS", "source": "nifty500_cache", "status": "ok" if close_rows else "missing", "close_rows": close_rows, "error": ""})

    if missing_symbols:
        tickers = [f"{symbol}.NS" for symbol in missing_symbols]
        ticker_to_symbol = dict(zip(tickers, missing_symbols))
        for start in range(0, len(tickers), chunk_size):
            chunk_tickers = tickers[start:start + chunk_size]
            chunk_symbols = [ticker_to_symbol[ticker] for ticker in chunk_tickers]
            print(f"Downloading expanded-parent chunk {start // chunk_size + 1}: {chunk_symbols[0]} ... {chunk_symbols[-1]} ({len(chunk_symbols)} names)")
            try:
                raw = yf.download(
                    tickers=chunk_tickers,
                    start=START_DATE,
                    end=END_DATE,
                    auto_adjust=False,
                    group_by="ticker",
                    threads=True,
                    progress=False,
                )
                for field, yf_field in {
                    "open": "Open",
                    "high": "High",
                    "low": "Low",
                    "close": "Close",
                    "adj_close": "Adj Close",
                    "volume": "Volume",
                }.items():
                    extracted = extract_yf_field(raw, chunk_tickers, ticker_to_symbol, yf_field)
                    field_parts[field].append(extracted)
                close_part = extract_yf_field(raw, chunk_tickers, ticker_to_symbol, "Close")
                for symbol in chunk_symbols:
                    non_null = int(close_part[symbol].notna().sum()) if symbol in close_part else 0
                    report_rows.append({"symbol": symbol, "ticker": f"{symbol}.NS", "source": "download", "status": "ok" if non_null else "missing", "close_rows": non_null, "error": ""})
            except Exception as exc:
                for symbol in chunk_symbols:
                    report_rows.append({"symbol": symbol, "ticker": f"{symbol}.NS", "source": "download", "status": "error", "close_rows": 0, "error": repr(exc)})
            time.sleep(0.25)

    data = {}
    for field, parts in field_parts.items():
        non_empty_parts = [part for part in parts if not part.empty]
        frame = pd.concat(non_empty_parts, axis=1) if non_empty_parts else pd.DataFrame(index=n500_adj_close.index)
        frame = frame.loc[:, ~frame.columns.duplicated()].sort_index()
        frame = frame.reindex(columns=symbols)
        write_frame(frame, EXPANDED_FIELD_FILES[field])
        data[field] = frame

    report = pd.DataFrame(report_rows).merge(source_map[["Symbol", "source_indices", "source_slugs"]], left_on="symbol", right_on="Symbol", how="left").drop(columns=["Symbol"], errors="ignore")
    report.to_csv(EXPANDED_DOWNLOAD_REPORT_PATH, index=False)
    display(report["status"].value_counts(dropna=False).to_frame("symbols"))
    return data


expanded_data = download_expanded_parent_ohlcv(expanded_symbols, refresh=EXPANDED_PARENT_PRICE_REFRESH, chunk_size=EXPANDED_DOWNLOAD_CHUNK_SIZE)
expanded_open = expanded_data["open"]
expanded_high = expanded_data["high"]
expanded_low = expanded_data["low"]
expanded_close = expanded_data["close"]
expanded_adj_close = expanded_data["adj_close"]
expanded_volume = expanded_data["volume"]

expanded_high_vol_mask_100, expanded_high_vol_universe_report = build_dynamic_high_vol_universe(
    expanded_adj_close,
    expanded_close,
    expanded_volume,
    basket_size=HIGH_VOL_BASKET_SIZE,
    buffer_size=HIGH_VOL_BUFFER_SIZE,
    vol_lookback=HIGH_VOL_LOOKBACK,
    min_history=HIGH_VOL_MIN_HISTORY,
    min_price=HIGH_VOL_MIN_PRICE,
    min_adv_rupees=HIGH_VOL_MIN_ADV_RUPEES,
)
expanded_high_vol_mask_100.astype(int).to_csv(EXPANDED_UNIVERSE_MASK_PATH)
expanded_high_vol_universe_report.to_csv(EXPANDED_UNIVERSE_REPORT_PATH, index=False)

expanded_returns = expanded_adj_close.pct_change(fill_method=None)
expanded_forward_returns = {h: forward_return(expanded_adj_close, h) for h in FORWARD_HORIZONS}
expanded_next_returns = expanded_adj_close.pct_change(fill_method=None).shift(-1)
expanded_raw_alpha = compute_alpha001_inside_universe(expanded_adj_close, expanded_returns, expanded_high_vol_mask_100)
expanded_alpha, expanded_direction, expanded_default_raw_ic, expanded_causal_direction, expanded_causal_train_mean_ic = causal_orient_signal(expanded_raw_alpha, expanded_forward_returns)
expanded_causal_direction.to_csv(ARTIFACT_DIR / "expanded_high_vol_top100_causal_orientation.csv")
expanded_weights = build_top_signal_weights(expanded_alpha, top_n=high_vol_primary_top_n, min_signal=0.0, rebalance_mask=weekly_reconstitution_mask(expanded_alpha.index))
expanded_benchmark_weights = equal_weight_universe_weights(expanded_high_vol_mask_100)

metric_parts = []
for panel_name, signal, fwd in [
    ("nifty500_high_vol_top100", high_vol_alpha, n500_forward_returns),
    ("expanded_high_vol_top100", expanded_alpha, expanded_forward_returns),
]:
    for horizon, fwd_frame in fwd.items():
        rank_ic = cross_sectional_corr_by_date(signal, fwd_frame, method="spearman")
        metric_parts.append(pd.DataFrame([{
            "panel": panel_name,
            "metric_type": "ic",
            "horizon_days": horizon,
            "mean_rank_ic": rank_ic.mean(),
            "median_rank_ic": rank_ic.median(),
            "rank_ic_vol": rank_ic.std(),
            "rank_icir": rank_ic.mean() / rank_ic.std() if rank_ic.std() and not pd.isna(rank_ic.std()) else np.nan,
            "positive_ic_rate": rank_ic.dropna().gt(0).mean(),
            "observations": rank_ic.notna().sum(),
        }]))
    buckets = bucket_returns(signal, fwd[DEFAULT_HORIZON], top_n=10)
    metric_parts.append(pd.DataFrame([{
        "panel": panel_name,
        "metric_type": "bucket",
        "horizon_days": DEFAULT_HORIZON,
        "top_n": 10,
        "mean_top_return": buckets["top_return"].mean(),
        "mean_bottom_return": buckets["bottom_return"].mean(),
        "mean_top_bottom_spread": buckets["spread_return"].mean(),
        "spread_hit_rate": buckets["spread_return"].gt(0).mean(),
        "observations": len(buckets),
    }]))

for cost_bps in [10.0, 20.0, 35.0]:
    for panel_name, weights, benchmark_weights, next_returns_frame in [
        ("nifty500_high_vol_top100", high_vol_primary_weights, high_vol_benchmark_weights, n500_next_returns),
        ("expanded_high_vol_top100", expanded_weights, expanded_benchmark_weights, expanded_next_returns),
    ]:
        alpha_bt = backtest_weights(weights, next_returns_frame, cost_bps=cost_bps)
        bench_bt = backtest_weights(benchmark_weights, next_returns_frame, cost_bps=cost_bps)
        excess = alpha_bt["returns"].sub(bench_bt["returns"], fill_value=0.0)
        for strategy_name, metrics in [
            ("alpha_top_bucket", alpha_bt["metrics"]),
            ("equal_weight_high_vol_universe", bench_bt["metrics"]),
            ("alpha_minus_high_vol_equal_weight", performance_metrics(excess)),
        ]:
            row = {"panel": panel_name, "metric_type": "strategy_cost_sensitivity", "strategy": strategy_name, "cost_bps": cost_bps}
            row.update(metrics)
            metric_parts.append(pd.DataFrame([row]))

expanded_high_vol_alpha001_metrics = pd.concat(metric_parts, ignore_index=True, sort=False)
expanded_high_vol_alpha001_metrics.to_csv(EXPANDED_ALPHA_METRICS_PATH, index=False)

latest_expanded_date = expanded_alpha.dropna(how="all").index[-1]
latest_expanded_scores = expanded_alpha.loc[latest_expanded_date].dropna().sort_values(ascending=False)
latest_expanded_selected = latest_expanded_scores.head(high_vol_primary_top_n)
latest_expanded_vol20 = expanded_returns.rolling(HIGH_VOL_LOOKBACK, min_periods=HIGH_VOL_LOOKBACK).std().shift(1) * math.sqrt(ANNUALIZATION)
latest_expanded_adv20 = expanded_close.mul(expanded_volume).rolling(20, min_periods=20).mean().shift(1)
if "high_vol_latest_candidates" not in globals() and (ARTIFACT_DIR / "high_vol_latest_candidates.csv").exists():
    high_vol_latest_candidates = pd.read_csv(ARTIFACT_DIR / "high_vol_latest_candidates.csv")
latest_nifty500_members = set(high_vol_latest_candidates["symbol"]) if "high_vol_latest_candidates" in globals() and not high_vol_latest_candidates.empty else set()
latest_rows = []
for symbol, score in latest_expanded_selected.items():
    source_info = source_map.set_index("Symbol").reindex([symbol]).iloc[0]
    latest_rows.append({
        "date": latest_expanded_date,
        "symbol": symbol,
        "alpha": "alpha001_expanded_high_vol_top100",
        "suggested_direction": "long",
        "target_weight_equal_weight": 1.0 / len(latest_expanded_selected) if len(latest_expanded_selected) else np.nan,
        "oriented_signal_score": score,
        "signal_orientation": "raw" if expanded_direction == 1 else "inverted",
        "active_expanded_high_vol_member": bool(expanded_high_vol_mask_100.loc[latest_expanded_date, symbol]),
        "also_in_latest_nifty500_high_vol_candidates": symbol in latest_nifty500_members,
        "in_nifty500_price_cache": symbol in n500_adj_close.columns,
        "lagged_20d_vol_ann": latest_expanded_vol20.loc[latest_expanded_date, symbol] if symbol in latest_expanded_vol20.columns else np.nan,
        "lagged_20d_adv_rupees": latest_expanded_adv20.loc[latest_expanded_date, symbol] if symbol in latest_expanded_adv20.columns else np.nan,
        "source_indices": source_info.get("source_indices"),
        "source_slugs": source_info.get("source_slugs"),
    })
expanded_high_vol_latest_candidates = pd.DataFrame(latest_rows)
expanded_high_vol_latest_candidates.to_csv(EXPANDED_LATEST_CANDIDATES_PATH, index=False)

expanded_counts = expanded_high_vol_mask_100.sum(axis=1)
expanded_post = expanded_high_vol_universe_report.query("member_count > 0") if not expanded_high_vol_universe_report.empty else pd.DataFrame()
expanded_summary = pd.Series({
    "expanded_parent_symbols": len(expanded_symbols),
    "symbols_with_any_adjusted_close": int(expanded_adj_close.notna().any().sum()),
    "new_symbols_beyond_nifty500_cache": len(set(expanded_symbols) - set(n500_adj_close.columns)),
    "first_non_empty_date": expanded_counts[expanded_counts > 0].index.min() if expanded_counts.gt(0).any() else pd.NaT,
    "median_member_count_after_warmup": expanded_counts[expanded_counts > 0].median(),
    "latest_eligible_count": int(expanded_post["eligible_count"].iloc[-1]) if not expanded_post.empty else 0,
    "avg_weekly_membership_turnover": expanded_post["membership_turnover_fraction"].mean() if not expanded_post.empty else np.nan,
})
print("Expanded high-vol parent summary")
display(expanded_summary.to_frame("expanded_parent"))
print("Expanded high-vol metrics")
display(expanded_high_vol_alpha001_metrics)
print("Latest expanded high-vol candidates")
display(expanded_high_vol_latest_candidates)


## 20. All-Stock Alpha#1 Weighting

The top-bucket test intentionally concentrates the signal. This section asks what happens if we avoid a hard bucket and use every active stock in the high-volatility basket. It compares equal-weight, top-10, all-name rank weighting, all-name score tilting, positive-score weighting, and full long-short score weighting across both the NIFTY500 high-vol basket and the expanded high-vol basket.

In [ ]:

ALL_STOCK_WEIGHTING_METRICS_PATH = ARTIFACT_DIR / "all_stock_alpha001_weighting_metrics.csv"


def build_all_stock_alpha_weights(signal: pd.DataFrame, mode: str, rebalance_mask: pd.Series | None = None) -> pd.DataFrame:
    rebalance_mask = weekly_reconstitution_mask(signal.index) if rebalance_mask is None else rebalance_mask.reindex(signal.index).fillna(False).astype(bool)
    weights = pd.DataFrame(0.0, index=signal.index, columns=signal.columns)
    last_weights = pd.Series(0.0, index=signal.columns)
    for i, date in enumerate(signal.index):
        should_rebalance = bool(rebalance_mask.loc[date]) or i == 0
        if not should_rebalance:
            weights.loc[date] = last_weights
            continue

        scores = signal.loc[date].dropna().astype(float)
        last_weights = pd.Series(0.0, index=signal.columns)
        if scores.empty:
            weights.loc[date] = last_weights
            continue

        if mode == "rank_weight_long_only":
            raw = (scores + 0.5).clip(lower=0.0)
            next_weights = raw / raw.sum() if raw.sum() > 0 else pd.Series(1.0 / len(scores), index=scores.index)
        elif mode == "score_tilt_long_only":
            sigma = scores.std(ddof=0)
            z = (scores - scores.mean()) / (sigma if sigma and sigma > 0 else 1.0)
            raw = (1.0 + 0.50 * z).clip(lower=0.05)
            next_weights = raw / raw.sum()
        elif mode == "positive_score_long_only":
            raw = scores.clip(lower=0.0)
            next_weights = raw / raw.sum() if raw.sum() > 0 else pd.Series(1.0 / len(scores), index=scores.index)
        elif mode == "all_stock_long_short":
            centered = scores - scores.mean()
            gross = centered.abs().sum()
            next_weights = centered / gross if gross > 0 else pd.Series(0.0, index=scores.index)
        else:
            raise ValueError(f"Unknown all-stock weighting mode: {mode}")

        last_weights.loc[next_weights.index] = next_weights
        weights.loc[date] = last_weights
    return weights


def summarize_weighted_strategy(panel: str, strategy: str, weights: pd.DataFrame, next_returns_frame: pd.DataFrame, cost_bps: float) -> dict:
    bt = backtest_weights(weights, next_returns_frame, cost_bps=cost_bps)
    row = {
        "panel": panel,
        "strategy": strategy,
        "cost_bps": cost_bps,
        "avg_names": weights.ne(0).sum(axis=1).replace(0, np.nan).mean(),
        "avg_abs_names": weights.abs().gt(1e-12).sum(axis=1).replace(0, np.nan).mean(),
        "avg_net_exposure": weights.sum(axis=1).replace(0, np.nan).mean(),
        "avg_gross_exposure": weights.abs().sum(axis=1).replace(0, np.nan).mean(),
    }
    row.update(bt["metrics"])
    return row


all_stock_panels = {
    "nifty500_high_vol_top100": {
        "signal": high_vol_alpha,
        "mask": high_vol_mask_100,
        "next_returns": n500_next_returns,
        "top_bucket_weights": high_vol_primary_weights,
    },
    "expanded_high_vol_top100": {
        "signal": expanded_alpha,
        "mask": expanded_high_vol_mask_100,
        "next_returns": expanded_next_returns,
        "top_bucket_weights": expanded_weights,
    },
}

all_stock_rows = []
for panel, cfg in all_stock_panels.items():
    signal = cfg["signal"]
    mask = cfg["mask"].reindex_like(signal).fillna(False)
    strategy_weights = {
        "equal_weight_all_active": equal_weight_universe_weights(mask),
        "top10_bucket_current": cfg["top_bucket_weights"],
        "rank_weight_all_long_only": build_all_stock_alpha_weights(signal, "rank_weight_long_only"),
        "score_tilt_all_long_only": build_all_stock_alpha_weights(signal, "score_tilt_long_only"),
        "positive_score_all_long_only": build_all_stock_alpha_weights(signal, "positive_score_long_only"),
        "all_stock_long_short": build_all_stock_alpha_weights(signal, "all_stock_long_short"),
    }
    for strategy, weights in strategy_weights.items():
        for cost_bps in [10.0, 20.0, 35.0]:
            all_stock_rows.append(summarize_weighted_strategy(panel, strategy, weights, cfg["next_returns"], cost_bps))

all_stock_alpha001_weighting_metrics = pd.DataFrame(all_stock_rows)
all_stock_alpha001_weighting_metrics.to_csv(ALL_STOCK_WEIGHTING_METRICS_PATH, index=False)

print("All-stock Alpha#1 weighting comparison at 20 bps")
display(
    all_stock_alpha001_weighting_metrics
    .query("cost_bps == 20.0")
    .sort_values(["panel", "sharpe"], ascending=[True, False])
)


## 21. India VIX and Index-Informed Volatility Gates

This section turns the index-monitoring idea into causal gates. India VIX is used when available, and each volatile index basket contributes observable stress features: rolling volatility, 20-day momentum, and breadth. The test asks whether Alpha#1 adds value over the equal-weight high-vol universe when those gates are active.

In [ ]:

INDIA_VIX_PATH = NIFTY500_DATA_DIR / "india_vix_close.csv"
VOLATILE_INDEX_GATE_REPORT_PATH = ARTIFACT_DIR / "volatile_index_regime_gate_report.csv"
VOLATILE_INDEX_REGIME_FEATURES_PATH = ARTIFACT_DIR / "volatile_index_regime_features.csv"
REFRESH_INDIA_VIX = False


def extract_single_close(raw_data: pd.DataFrame, ticker: str) -> pd.Series | None:
    if raw_data.empty:
        return None
    if isinstance(raw_data.columns, pd.MultiIndex):
        for key in [(ticker, "Close"), ("Close", ticker), (ticker, "Adj Close"), ("Adj Close", ticker)]:
            if key in raw_data.columns:
                return pd.to_numeric(raw_data[key], errors="coerce")
    for field in ["Close", "Adj Close"]:
        if field in raw_data.columns:
            return pd.to_numeric(raw_data[field], errors="coerce")
    return None


def load_india_vix(refresh: bool = False) -> pd.Series:
    if INDIA_VIX_PATH.exists() and not refresh:
        frame = read_frame(INDIA_VIX_PATH)
        series = frame.iloc[:, 0].rename("india_vix")
        print(f"Loaded cached India VIX: {series.dropna().shape[0]} rows")
        return series
    errors = []
    for ticker in ["^INDIAVIX", "INDIAVIX.NS"]:
        try:
            raw = yf.download(ticker, start=START_DATE, end=END_DATE, auto_adjust=False, progress=False, threads=False)
            series = extract_single_close(raw, ticker)
            if series is None or series.dropna().empty:
                raise ValueError("no close series returned")
            series.index = pd.to_datetime(series.index).tz_localize(None)
            series = series.rename("india_vix").sort_index()
            write_frame(series, INDIA_VIX_PATH)
            print(f"Downloaded India VIX using {ticker}: {series.dropna().shape[0]} rows")
            return series
        except Exception as exc:
            errors.append({"ticker": ticker, "error": repr(exc)})
    pd.DataFrame(errors).to_csv(ARTIFACT_DIR / "india_vix_download_errors.csv", index=False)
    print("India VIX download failed; continuing with index-derived gates only")
    display(pd.DataFrame(errors))
    return pd.Series(dtype=float, name="india_vix")


def compounded_return(series: pd.Series, window: int) -> pd.Series:
    return (1.0 + series).rolling(window, min_periods=window).apply(np.prod, raw=True) - 1.0


def breadth_for_mask(mask: pd.DataFrame, returns_frame: pd.DataFrame) -> pd.Series:
    masked = returns_frame.where(mask.reindex_like(returns_frame).fillna(False))
    count = masked.notna().sum(axis=1).replace(0, np.nan)
    return masked.gt(0).sum(axis=1).div(count).rolling(20, min_periods=20).mean()


india_vix = load_india_vix(refresh=REFRESH_INDIA_VIX)
volatile_index_regime_features = pd.DataFrame(index=n500_returns.index)

if not india_vix.empty:
    vix = india_vix.reindex(volatile_index_regime_features.index).ffill()
    volatile_index_regime_features["india_vix_level"] = rolling_tercile_label(vix, "low", "middle", "high")
    volatile_index_regime_features["india_vix_change"] = rolling_tercile_label(vix.pct_change(5, fill_method=None), "falling", "stable", "rising")

volatile_index_proxy_returns = {}
for slug, mask in volatile_index_masks.items():
    index_return = ew_returns_for_mask(mask, n500_returns).replace([np.inf, -np.inf], np.nan)
    volatile_index_proxy_returns[slug] = index_return
    vol20 = index_return.rolling(20, min_periods=20).std() * math.sqrt(ANNUALIZATION)
    mom20 = compounded_return(index_return, 20)
    breadth20 = breadth_for_mask(mask, n500_returns)
    volatile_index_regime_features[f"{slug}_vol_level"] = rolling_tercile_label(vol20, "low", "middle", "high")
    volatile_index_regime_features[f"{slug}_mom20"] = rolling_tercile_label(mom20, "weak", "middle", "strong")
    volatile_index_regime_features[f"{slug}_breadth"] = rolling_tercile_label(breadth20, "weak", "middle", "broad")

candidate_index_gates = {"ungated": pd.Series(True, index=high_vol_alpha.index)}
if "india_vix_level" in volatile_index_regime_features:
    candidate_index_gates["india_vix_high"] = volatile_index_regime_features["india_vix_level"].eq("high")
    candidate_index_gates["india_vix_rising"] = volatile_index_regime_features["india_vix_change"].eq("rising")
    candidate_index_gates["india_vix_high_or_rising"] = candidate_index_gates["india_vix_high"] | candidate_index_gates["india_vix_rising"]

for slug in ["nifty_high_beta_50", "nifty_midcap_150", "nifty_smallcap_250", "nifty_it", "nifty_bank"]:
    if f"{slug}_vol_level" in volatile_index_regime_features:
        candidate_index_gates[f"{slug}_vol_high"] = volatile_index_regime_features[f"{slug}_vol_level"].eq("high")
        candidate_index_gates[f"{slug}_mom20_weak"] = volatile_index_regime_features[f"{slug}_mom20"].eq("weak")
        candidate_index_gates[f"{slug}_breadth_weak"] = volatile_index_regime_features[f"{slug}_breadth"].eq("weak")

vol_high_cols = [col for col in volatile_index_regime_features.columns if col.endswith("_vol_level")]
breadth_cols = [col for col in volatile_index_regime_features.columns if col.endswith("_breadth")]
if vol_high_cols:
    candidate_index_gates["any_volatile_index_vol_high"] = volatile_index_regime_features[vol_high_cols].eq("high").any(axis=1)
if breadth_cols:
    candidate_index_gates["any_volatile_index_breadth_weak"] = volatile_index_regime_features[breadth_cols].eq("weak").any(axis=1)
composite_parts = []
for name in ["india_vix_high_or_rising", "any_volatile_index_vol_high", "any_volatile_index_breadth_weak"]:
    if name in candidate_index_gates:
        composite_parts.append(candidate_index_gates[name].reindex(high_vol_alpha.index).fillna(False))
if composite_parts:
    candidate_index_gates["vix_or_index_stress"] = pd.concat(composite_parts, axis=1).any(axis=1)

index_gate_rows = [
    metrics_for_gate(name, gate, high_vol_primary_weights, high_vol_benchmark_weights, n500_next_returns, cost_bps=20.0)
    for name, gate in candidate_index_gates.items()
]
volatile_index_strategy_gate_report = pd.DataFrame(index_gate_rows)
volatile_index_strategy_gate_report.insert(0, "report_type", "gated_strategy")

selected_feature_cols = [
    col for col in volatile_index_regime_features.columns
    if col in {"india_vix_level", "india_vix_change"}
    or col.endswith("_vol_level")
    or col.endswith("_mom20")
    or col.endswith("_breadth")
]
volatile_index_conditional_info = conditional_alpha_information_report(
    volatile_index_regime_features[selected_feature_cols],
    high_vol_alpha,
    n500_forward_returns[DEFAULT_HORIZON],
    high_vol_spread,
    primary_bt_20["returns"],
    benchmark_bt_20["returns"],
) if selected_feature_cols else pd.DataFrame()
if not volatile_index_conditional_info.empty:
    volatile_index_conditional_info.insert(0, "report_type", "conditional_information")

volatile_index_gate_report = pd.concat([volatile_index_strategy_gate_report, volatile_index_conditional_info], ignore_index=True, sort=False)
volatile_index_gate_report.to_csv(VOLATILE_INDEX_GATE_REPORT_PATH, index=False)
volatile_index_regime_features.to_csv(VOLATILE_INDEX_REGIME_FEATURES_PATH)

print("Index-informed gate strategy comparison")
display(volatile_index_strategy_gate_report.sort_values("excess_sharpe", ascending=False))
print("Best index-informed conditional information pockets")
if not volatile_index_conditional_info.empty:
    display(volatile_index_conditional_info.sort_values("mean_rank_ic", ascending=False).head(25))


## 22. Full Verification and Failure Discovery

This block formalizes the discard decision. It does not download new data or use synthetic data. It verifies the Alpha#1 math, recomputes core metrics from cached artifacts, removes full-sample orientation from tradable high-vol signals, and makes active return versus the relevant equal-weight universe the primary portfolio test.

In [ ]:

VERIFICATION_ARTIFACTS = {
    "formula": ARTIFACT_DIR / "alpha001_formula_validation.csv",
    "metric": ARTIFACT_DIR / "alpha001_metric_validation.csv",
    "causality": ARTIFACT_DIR / "alpha001_causality_audit.csv",
    "corrected_ic": ARTIFACT_DIR / "alpha001_corrected_ic_summary.csv",
    "gross_net": ARTIFACT_DIR / "alpha001_gross_net_attribution.csv",
    "active_benchmark": ARTIFACT_DIR / "alpha001_active_vs_benchmark_report.csv",
    "failure_stage": ARTIFACT_DIR / "alpha001_failure_stage_report.csv",
    "decision": ARTIFACT_DIR / "alpha001_discard_decision_report.csv",
}


def strict_sum_portfolio_returns(weights: pd.DataFrame, next_returns_frame: pd.DataFrame) -> pd.Series:
    aligned = next_returns_frame.reindex_like(weights)
    valid_exposure_return = weights.abs().gt(0) & aligned.notna()
    no_exposure = weights.abs().sum(axis=1).eq(0)
    return weights.mul(aligned).sum(axis=1, min_count=1).where(valid_exposure_return.any(axis=1) | no_exposure)


def strict_backtest(weights: pd.DataFrame, next_returns_frame: pd.DataFrame, cost_bps: float) -> dict:
    gross = strict_sum_portfolio_returns(weights, next_returns_frame)
    turnover = weights.diff().abs().sum(axis=1, min_count=1).fillna(weights.abs().sum(axis=1))
    costs = turnover * (cost_bps / 10000.0)
    net = gross - costs
    valid_index = net.dropna().index
    return {
        "returns": net.reindex(valid_index),
        "gross_returns": gross.reindex(valid_index),
        "turnover": turnover.reindex(valid_index).fillna(0.0),
        "costs": costs.reindex(valid_index).fillna(0.0),
        "metrics": performance_metrics(net, turnover),
    }


def literal_alpha001(close_frame: pd.DataFrame, return_frame: pd.DataFrame, active_mask: pd.DataFrame | None = None, convention: str = "oldest_to_newest") -> pd.DataFrame:
    vol20 = return_frame.rolling(20, min_periods=20).std()
    condition_value = close_frame.where(~return_frame.lt(0), vol20)
    transformed = signed_power(condition_value, 2)
    argmax5 = ts_argmax_position(transformed, 5, convention=convention)
    if active_mask is not None:
        argmax5 = argmax5.where(active_mask.reindex_like(argmax5).fillna(False))
    return rank_cross_sectional(argmax5) - 0.5


def validation_row(check: str, passed: bool, detail: str, observed=np.nan, expected=np.nan) -> dict:
    return {"check": check, "passed": bool(passed), "observed": observed, "expected": expected, "detail": detail}


def close_enough(a, b, tol: float = 1e-10) -> bool:
    if pd.isna(a) and pd.isna(b):
        return True
    return bool(abs(float(a) - float(b)) <= tol)

formula_rows = []
fixture = pd.DataFrame({"x": [-2.0, 0.0, 3.0]})
formula_rows.append(validation_row("signed_power_square", signed_power(fixture, 2)["x"].tolist() == [-4.0, 0.0, 9.0], "sign(x) * abs(x)^2 keeps negative sign"))
argmax_fixture = pd.DataFrame({"x": [1.0, 3.0, 2.0, 5.0, 4.0]})
argmax_oldest = ts_argmax_position(argmax_fixture, 5, "oldest_to_newest").iloc[-1, 0]
argmax_days_since = ts_argmax_position(argmax_fixture, 5, "days_since_max").iloc[-1, 0]
formula_rows.append(validation_row("ts_argmax_oldest_to_newest_fixture", argmax_oldest == 4.0, "[1,3,2,5,4] max is fourth item under oldest-to-newest indexing", argmax_oldest, 4.0))
formula_rows.append(validation_row("ts_argmax_days_since_fixture", argmax_days_since == 2.0, "[1,3,2,5,4] max is two days back under days-since-max indexing", argmax_days_since, 2.0))
tie_fixture = pd.DataFrame({"x": [1.0, 5.0, 5.0, 2.0, 3.0]})
tie_oldest = ts_argmax_position(tie_fixture, 5, "oldest_to_newest").iloc[-1, 0]
formula_rows.append(validation_row("ts_argmax_tie_policy_first_max", tie_oldest == 2.0, "np.nanargmax chooses the first tied maximum", tie_oldest, 2.0))
rank_fixture = pd.DataFrame([[10.0, 20.0, 20.0, 40.0]], columns=list("ABCD"))
rank_observed = (rank_cross_sectional(rank_fixture) - 0.5).iloc[0].round(6).tolist()
formula_rows.append(validation_row("rank_pct_average_tie_centering", rank_observed == [-0.25, 0.125, 0.125, 0.5], "pandas pct rank uses average ties and rank/n scaling", rank_observed, [-0.25, 0.125, 0.125, 0.5]))
price_fixture = pd.DataFrame({"A": [100.0, 110.0, 121.0]})
fwd_fixture = forward_return(price_fixture, 2).iloc[0, 0]
formula_rows.append(validation_row("forward_return_two_period_fixture", close_enough(fwd_fixture, 0.21), "price.shift(-h)/price - 1", fwd_fixture, 0.21))
weights_fixture = pd.DataFrame({"A": [1.0, 0.0], "B": [0.0, 1.0]}, index=pd.date_range("2024-01-01", periods=2))
returns_fixture = pd.DataFrame({"A": [0.01, 0.02], "B": [0.00, 0.03]}, index=weights_fixture.index)
turnover_fixture = weights_fixture.diff().abs().sum(axis=1, min_count=1).fillna(weights_fixture.abs().sum(axis=1)).tolist()
formula_rows.append(validation_row("turnover_fixture", turnover_fixture == [1.0, 2.0], "initial gross exposure then absolute weight change", turnover_fixture, [1.0, 2.0]))
bt_fixture = strict_backtest(weights_fixture, returns_fixture, cost_bps=10.0)
formula_rows.append(validation_row("cost_fixture", close_enough(bt_fixture["costs"].iloc[1], 0.002), "cost = turnover * bps / 10000", bt_fixture["costs"].iloc[1], 0.002))
perf_fixture = performance_metrics(pd.Series([0.01, -0.005], index=pd.date_range("2024-01-01", periods=2)))
formula_rows.append(validation_row("performance_fixture_observations", perf_fixture["observations"] == 2, "performance_metrics drops NaN and counts realized observations", perf_fixture["observations"], 2))

# Formula fidelity variants on the baseline universe.
formula_adj_literal = literal_alpha001(return_price, returns)
formula_raw_literal = literal_alpha001(raw_close, raw_close.pct_change(fill_method=None))
formula_current_corr = formula_adj_literal.stack().rank().corr(raw_alpha001.reindex_like(formula_adj_literal).stack().rank())
formula_raw_corr = formula_raw_literal.stack().rank().corr(raw_alpha001.reindex_like(formula_raw_literal).stack().rank())
formula_rows.append(validation_row("literal_adjusted_vs_current_rank_corr", formula_current_corr > 0.98, "literal adjusted-close formula should remain close to current cached raw Alpha#1", formula_current_corr, "> 0.98"))
formula_rows.append(validation_row("literal_raw_close_variant_available", formula_raw_literal.notna().sum().sum() > 0, "raw-close formula-fidelity variant was computed", int(formula_raw_literal.notna().sum().sum()), "> 0"))
formula_rows.append(validation_row("raw_close_vs_current_rank_corr_report_only", True, "raw-close variant rank correlation to cached current formula; not a pass/fail criterion", formula_raw_corr, "report"))

formula_validation = pd.DataFrame(formula_rows)
formula_validation.to_csv(VERIFICATION_ARTIFACTS["formula"], index=False)

# Reproducibility checks against existing artifacts.
metric_rows = []
for artifact_name, signal_frame, price_frame in [
    ("raw_ic_summary.csv", raw_alpha001, return_price),
    ("oriented_ic_summary.csv", oriented_alpha001, return_price),
]:
    stored = pd.read_csv(ARTIFACT_DIR / artifact_name).set_index("horizon_days")
    diffs = []
    for horizon in FORWARD_HORIZONS:
        ic = cross_sectional_corr_by_date(signal_frame, forward_return(price_frame, horizon), method="spearman")
        diffs.append(abs(ic.mean() - stored.loc[horizon, "mean_rank_ic"]))
    metric_rows.append(validation_row(f"reproduce_{artifact_name}", max(diffs) < 1e-10, "mean Rank IC reproduces from cached signal and price data", max(diffs), "< 1e-10"))

stored_buckets = pd.read_csv(ARTIFACT_DIR / "bucket_summary.csv")
for horizon, top_n in [(1, 10), (5, 5), (5, 10)]:
    frame = bucket_returns(oriented_alpha001, forward_return(return_price, horizon), top_n=top_n)
    row = stored_buckets[(stored_buckets["horizon_days"].eq(horizon)) & (stored_buckets["top_n"].eq(top_n))].iloc[0]
    spread_diff = abs(frame["spread_return"].mean() - row["mean_spread_return"])
    metric_rows.append(validation_row(f"reproduce_bucket_h{horizon}_top{top_n}", spread_diff < 1e-10, "bucket spread reproduces from cached signal and price data", spread_diff, "< 1e-10"))

stored_simple = pd.read_csv(ARTIFACT_DIR / "simple_strategy_metrics.csv").set_index("strategy")
for name, top_n, long_short in [("long_only_top5", 5, False), ("long_only_top10", 10, False), ("long_short_top5", 5, True), ("long_short_top10", 10, True)]:
    weights = build_weights(oriented_alpha001, top_n=top_n, long_short=long_short)
    bt = strict_backtest(weights, next_session_returns, cost_bps=COST_BPS_PER_TURNOVER)
    stored_obs = int(stored_simple.loc[name, "observations"])
    metric_rows.append(validation_row(f"strict_backtest_{name}_has_no_final_nan_day", bt["metrics"]["observations"] <= stored_obs, "strict backtest excludes all-NaN final return rows", bt["metrics"]["observations"], f"<= {stored_obs}"))

metric_validation = pd.DataFrame(metric_rows)
metric_validation.to_csv(VERIFICATION_ARTIFACTS["metric"], index=False)

display(formula_validation)
display(metric_validation)


## 23. Causality, Corrected IC, and Active Benchmark Reports

These reports use cached real data and the corrected conventions from the verification section: literal conditional semantics, corrected positive IC denominators, strict return rows, Sortino everywhere, and causal rolling orientation for tradable high-vol signals.

In [ ]:


def corrected_ic_rows(panel: str, signal: pd.DataFrame, forward_by_horizon: dict[int, pd.DataFrame]) -> list[dict]:
    rows = []
    for horizon, future in forward_by_horizon.items():
        rank_ic = cross_sectional_corr_by_date(signal, future, method="spearman")
        vol = rank_ic.std()
        rows.append({
            "panel": panel,
            "horizon_days": horizon,
            "mean_rank_ic": rank_ic.mean(),
            "median_rank_ic": rank_ic.median(),
            "rank_ic_vol": vol,
            "rank_icir": rank_ic.mean() / vol if vol and not pd.isna(vol) else np.nan,
            "positive_ic_rate_corrected": rank_ic.dropna().gt(0).mean(),
            "positive_ic_rate_old_nan_as_false": rank_ic.gt(0).mean(),
            "observations": rank_ic.notna().sum(),
        })
    return rows


def latest_orientation_label(direction: int) -> str:
    return "raw" if int(direction) == 1 else "inverted"

corrected_ic = []
corrected_ic += corrected_ic_rows("nifty50_raw_alpha001", raw_alpha001, forward_returns)
corrected_ic += corrected_ic_rows("nifty50_oriented_full_sample_research", oriented_alpha001, forward_returns)
corrected_ic += corrected_ic_rows("nifty500_all_eligible_causal", n500_all_alpha, n500_forward_returns)
corrected_ic += corrected_ic_rows("nifty500_high_vol_top100_causal", high_vol_alpha, n500_forward_returns)
if "expanded_alpha" in globals():
    corrected_ic += corrected_ic_rows("expanded_high_vol_top100_causal", expanded_alpha, expanded_forward_returns)

corrected_ic_summary = pd.DataFrame(corrected_ic)
corrected_ic_summary.to_csv(VERIFICATION_ARTIFACTS["corrected_ic"], index=False)

causality_rows = []
for panel, direction_series, train_mean_series, scalar_direction, raw_ic in [
    ("nifty500_all_eligible", n500_all_causal_direction, n500_all_causal_train_mean_ic, n500_all_direction, n500_all_default_raw_ic),
    ("nifty500_high_vol_top100", high_vol_causal_direction, high_vol_causal_train_mean_ic, high_vol_direction, high_vol_default_raw_ic),
]:
    available = direction_series.dropna()
    causality_rows.append({
        "panel": panel,
        "orientation_method": "rolling_training_only",
        "default_horizon_days": DEFAULT_HORIZON,
        "train_window_ic_days": 504,
        "min_ic_observations": 126,
        "full_sample_raw_mean_ic_for_audit": raw_ic,
        "latest_causal_orientation": latest_orientation_label(scalar_direction),
        "orientation_available_fraction": direction_series.notna().mean(),
        "inverted_fraction_when_available": available.eq(-1).mean() if len(available) else np.nan,
        "latest_train_mean_ic": train_mean_series.dropna().iloc[-1] if train_mean_series.notna().any() else np.nan,
        "candidate_uses_full_sample_orientation": False,
    })
if "expanded_causal_direction" in globals():
    available = expanded_causal_direction.dropna()
    causality_rows.append({
        "panel": "expanded_high_vol_top100",
        "orientation_method": "rolling_training_only",
        "default_horizon_days": DEFAULT_HORIZON,
        "train_window_ic_days": 504,
        "min_ic_observations": 126,
        "full_sample_raw_mean_ic_for_audit": expanded_default_raw_ic,
        "latest_causal_orientation": latest_orientation_label(expanded_direction),
        "orientation_available_fraction": expanded_causal_direction.notna().mean(),
        "inverted_fraction_when_available": available.eq(-1).mean() if len(available) else np.nan,
        "latest_train_mean_ic": expanded_causal_train_mean_ic.dropna().iloc[-1] if expanded_causal_train_mean_ic.notna().any() else np.nan,
        "candidate_uses_full_sample_orientation": False,
    })
causality_rows.extend([
    {"panel": "nifty50_baseline_raw", "orientation_method": "none_raw_formula", "candidate_uses_full_sample_orientation": False, "note": "raw published-style formula output"},
    {"panel": "nifty50_baseline_oriented", "orientation_method": "full_sample_research_only", "candidate_uses_full_sample_orientation": True, "note": "kept as baseline research diagnostic; walk-forward section uses training orientation"},
    {"panel": "regime_gates", "orientation_method": "same_day_close_observable", "candidate_uses_full_sample_orientation": False, "note": "causal for after-close to next-close; shift one day for pre-close execution"},
])
causality_audit = pd.DataFrame(causality_rows)
causality_audit.to_csv(VERIFICATION_ARTIFACTS["causality"], index=False)


def beta_corr_to_market(returns_series: pd.Series, market_returns: pd.Series) -> tuple[float, float]:
    pair = pd.concat([returns_series.rename("strategy"), market_returns.rename("market")], axis=1).dropna()
    if len(pair) < 30 or pair["market"].var(ddof=0) == 0:
        return np.nan, np.nan
    beta = pair["strategy"].cov(pair["market"]) / pair["market"].var()
    corr = pair["strategy"].corr(pair["market"])
    return beta, corr


def add_metric_prefix(metrics: dict, prefix: str) -> dict:
    return {f"{prefix}_{key}": value for key, value in metrics.items()}

market_returns = nifty_close.pct_change(fill_method=None) if "nifty_close" in globals() else pd.Series(dtype=float)

def gross_net_row(panel: str, strategy: str, weights: pd.DataFrame, next_returns_frame: pd.DataFrame, cost_bps: float) -> dict:
    bt = strict_backtest(weights, next_returns_frame, cost_bps=cost_bps)
    gross_metrics = performance_metrics(bt["gross_returns"], bt["turnover"])
    net_metrics = performance_metrics(bt["returns"], bt["turnover"])
    row = {
        "panel": panel,
        "strategy": strategy,
        "cost_bps": cost_bps,
        "avg_daily_cost": bt["costs"].mean(),
        "annualized_cost_drag_simple": bt["costs"].mean() * ANNUALIZATION,
        "gross_minus_net_cagr": gross_metrics.get("cagr", np.nan) - net_metrics.get("cagr", np.nan),
    }
    row.update(add_metric_prefix(gross_metrics, "gross"))
    row.update(add_metric_prefix(net_metrics, "net"))
    return row

portfolio_specs = []
portfolio_specs.append(("nifty50_current", "alpha_top10_daily", build_weights(oriented_alpha001, top_n=10, long_short=False), next_session_returns))
portfolio_specs.append(("nifty50_current", "equal_weight_active", equal_weight_universe_weights(oriented_alpha001.notna()), next_session_returns))
portfolio_specs.append(("nifty500_all_eligible", "alpha_top10_weekly", build_top_signal_weights(n500_all_alpha, top_n=10, min_signal=0.0, rebalance_mask=weekly_reconstitution_mask(n500_all_alpha.index)), n500_next_returns))
portfolio_specs.append(("nifty500_all_eligible", "equal_weight_active", equal_weight_universe_weights(n500_eligible_mask), n500_next_returns))
portfolio_specs.append(("nifty500_high_vol_top100", "alpha_top10_weekly", high_vol_primary_weights, n500_next_returns))
portfolio_specs.append(("nifty500_high_vol_top100", "equal_weight_active", high_vol_benchmark_weights, n500_next_returns))
if "expanded_alpha" in globals():
    portfolio_specs.append(("expanded_high_vol_top100", "alpha_top10_weekly", expanded_weights, expanded_next_returns))
    portfolio_specs.append(("expanded_high_vol_top100", "equal_weight_active", expanded_benchmark_weights, expanded_next_returns))
    portfolio_specs.append(("expanded_high_vol_top100", "score_tilt_all_long_only", build_all_stock_alpha_weights(expanded_alpha, "score_tilt_long_only"), expanded_next_returns))
portfolio_specs.append(("nifty500_high_vol_top100", "score_tilt_all_long_only", build_all_stock_alpha_weights(high_vol_alpha, "score_tilt_long_only"), n500_next_returns))
portfolio_specs.append(("nifty500_high_vol_top100", "positive_score_all_long_only", build_all_stock_alpha_weights(high_vol_alpha, "positive_score_long_only"), n500_next_returns))
portfolio_specs.append(("nifty500_high_vol_top100", "all_stock_long_short", build_all_stock_alpha_weights(high_vol_alpha, "all_stock_long_short"), n500_next_returns))

attribution_rows = []
for panel, strategy, weights, next_returns_frame in portfolio_specs:
    for cost_bps in [10.0, 20.0, 35.0]:
        attribution_rows.append(gross_net_row(panel, strategy, weights, next_returns_frame, cost_bps))
gross_net_attribution = pd.DataFrame(attribution_rows)
gross_net_attribution.to_csv(VERIFICATION_ARTIFACTS["gross_net"], index=False)

active_pairs = [
    ("nifty50_current", "alpha_top10_daily", build_weights(oriented_alpha001, top_n=10, long_short=False), "equal_weight_active", equal_weight_universe_weights(oriented_alpha001.notna()), next_session_returns),
    ("nifty500_all_eligible", "alpha_top10_weekly", build_top_signal_weights(n500_all_alpha, top_n=10, min_signal=0.0, rebalance_mask=weekly_reconstitution_mask(n500_all_alpha.index)), "equal_weight_active", equal_weight_universe_weights(n500_eligible_mask), n500_next_returns),
    ("nifty500_high_vol_top100", "alpha_top10_weekly", high_vol_primary_weights, "equal_weight_active", high_vol_benchmark_weights, n500_next_returns),
]
if "expanded_alpha" in globals():
    active_pairs.append(("expanded_high_vol_top100", "alpha_top10_weekly", expanded_weights, "equal_weight_active", expanded_benchmark_weights, expanded_next_returns))

active_rows = []
for panel, alpha_name, alpha_weights, benchmark_name, benchmark_weights, next_returns_frame in active_pairs:
    alpha_bt = strict_backtest(alpha_weights, next_returns_frame, cost_bps=20.0)
    benchmark_bt = strict_backtest(benchmark_weights, next_returns_frame, cost_bps=20.0)
    excess = alpha_bt["returns"].sub(benchmark_bt["returns"], fill_value=0.0)
    excess_metrics = performance_metrics(excess)
    alpha_beta, alpha_corr = beta_corr_to_market(alpha_bt["returns"], market_returns)
    benchmark_beta, benchmark_corr = beta_corr_to_market(benchmark_bt["returns"], market_returns)
    row = {"panel": panel, "alpha_strategy": alpha_name, "benchmark_strategy": benchmark_name, "cost_bps": 20.0}
    row.update(add_metric_prefix(alpha_bt["metrics"], "alpha"))
    row.update(add_metric_prefix(benchmark_bt["metrics"], "benchmark"))
    row.update(add_metric_prefix(excess_metrics, "excess"))
    row.update({
        "alpha_beta_to_nifty": alpha_beta,
        "alpha_corr_to_nifty": alpha_corr,
        "benchmark_beta_to_nifty": benchmark_beta,
        "benchmark_corr_to_nifty": benchmark_corr,
    })
    active_rows.append(row)

# Carry index-guided basket diagnostics into the active benchmark report; these are current-constituent diagnostics, not historical index-membership backtests.
volatile_metrics_path = ARTIFACT_DIR / "volatile_index_alpha001_metrics.csv"
if volatile_metrics_path.exists():
    volatile_metrics = pd.read_csv(volatile_metrics_path)
    index_excess = volatile_metrics.query("metric_type == 'strategy_20bps' and strategy == 'alpha_minus_equal_weight'").copy()
    for _, item in index_excess.iterrows():
        active_rows.append({
            "panel": item.get("panel"),
            "alpha_strategy": "alpha_top_bucket_current_constituents",
            "benchmark_strategy": "equal_weight_current_constituents",
            "cost_bps": item.get("cost_bps"),
            "excess_total_return": item.get("total_return"),
            "excess_cagr": item.get("cagr"),
            "excess_annual_vol": item.get("annual_vol"),
            "excess_sharpe": item.get("sharpe"),
            "excess_max_drawdown": item.get("max_drawdown"),
            "diagnostic_scope": "current_constituent_index_basket",
        })

active_vs_benchmark_report = pd.DataFrame(active_rows)
active_vs_benchmark_report.to_csv(VERIFICATION_ARTIFACTS["active_benchmark"], index=False)

display(corrected_ic_summary)
display(causality_audit)
display(active_vs_benchmark_report.sort_values("excess_sharpe", ascending=False, na_position="last").head(20))


## 24. Failure Attribution and Discard Decision

The final table classifies each stage as `pass`, `weak`, `fail`, or `unverified`. A standalone Alpha#1 promotion requires verified math, causal orientation, stable positive information, and positive active return versus the relevant equal-weight active universe after costs.

In [ ]:

active_core = active_vs_benchmark_report[active_vs_benchmark_report["panel"].isin(["nifty500_high_vol_top100", "expanded_high_vol_top100"])]
high_vol_active = active_vs_benchmark_report[active_vs_benchmark_report["panel"].eq("nifty500_high_vol_top100")]
expanded_active = active_vs_benchmark_report[active_vs_benchmark_report["panel"].eq("expanded_high_vol_top100")]
formula_pass = bool(formula_validation["passed"].all())
metric_pass = bool(metric_validation["passed"].all())
causal_pass = not bool(causality_audit.query("panel in ['nifty500_high_vol_top100', 'expanded_high_vol_top100']")["candidate_uses_full_sample_orientation"].fillna(False).any())
hv_excess_sharpe = high_vol_active["excess_sharpe"].dropna().iloc[0] if not high_vol_active.empty and high_vol_active["excess_sharpe"].notna().any() else np.nan
expanded_excess_sharpe = expanded_active["excess_sharpe"].dropna().iloc[0] if not expanded_active.empty and expanded_active["excess_sharpe"].notna().any() else np.nan
hv_5d_ic = corrected_ic_summary.query("panel == 'nifty500_high_vol_top100_causal' and horizon_days == @DEFAULT_HORIZON")["mean_rank_ic"].iloc[0]
expanded_5d_ic = corrected_ic_summary.query("panel == 'expanded_high_vol_top100_causal' and horizon_days == @DEFAULT_HORIZON")["mean_rank_ic"].iloc[0] if "expanded_high_vol_top100_causal" in set(corrected_ic_summary["panel"]) else np.nan

def classify_weak_positive(value: float, fail_if_negative: bool = False) -> str:
    if pd.isna(value):
        return "unverified"
    if fail_if_negative and value <= 0:
        return "fail"
    if value > 0.03:
        return "pass"
    if value > 0:
        return "weak"
    return "fail"

failure_rows = [
    {"stage": "formula_fixtures", "classification": "pass" if formula_pass else "fail", "evidence": f"{int(formula_validation['passed'].sum())}/{len(formula_validation)} formula checks passed"},
    {"stage": "metric_reproducibility", "classification": "pass" if metric_pass else "fail", "evidence": f"{int(metric_validation['passed'].sum())}/{len(metric_validation)} metric checks passed"},
    {"stage": "causal_orientation", "classification": "pass" if causal_pass else "fail", "evidence": "high-vol and expanded candidate paths use rolling training-only orientation"},
    {"stage": "ic_strength_high_vol", "classification": classify_weak_positive(hv_5d_ic), "evidence": f"NIFTY500 high-vol corrected 5d IC = {hv_5d_ic:.6f}"},
    {"stage": "ic_strength_expanded", "classification": classify_weak_positive(expanded_5d_ic), "evidence": f"expanded high-vol corrected 5d IC = {expanded_5d_ic:.6f}" if pd.notna(expanded_5d_ic) else "expanded panel unavailable"},
    {"stage": "portfolio_active_high_vol", "classification": classify_weak_positive(hv_excess_sharpe, fail_if_negative=True), "evidence": f"20 bps high-vol alpha-minus-EW Sharpe = {hv_excess_sharpe:.6f}" if pd.notna(hv_excess_sharpe) else "missing active report"},
    {"stage": "portfolio_active_expanded", "classification": classify_weak_positive(expanded_excess_sharpe, fail_if_negative=True), "evidence": f"20 bps expanded alpha-minus-EW Sharpe = {expanded_excess_sharpe:.6f}" if pd.notna(expanded_excess_sharpe) else "missing active report"},
    {"stage": "turnover_cost", "classification": "fail", "evidence": "top-bucket strategies have much higher turnover and cost drag than equal-weight active universes"},
    {"stage": "regime_gating", "classification": "weak", "evidence": "breadth-weak gates improve relative returns in some pockets but often have poor absolute alpha returns"},
    {"stage": "universe_expansion", "classification": "fail", "evidence": "expanded universe adds breadth but worsens top-bucket active performance after costs"},
]
failure_stage_report = pd.DataFrame(failure_rows)
failure_stage_report.to_csv(VERIFICATION_ARTIFACTS["failure_stage"], index=False)

if formula_pass and metric_pass and causal_pass and pd.notna(hv_excess_sharpe) and hv_excess_sharpe > 0.25:
    decision = "promote_candidate"
elif formula_pass and metric_pass and causal_pass and pd.notna(hv_5d_ic) and hv_5d_ic > 0 and (pd.isna(hv_excess_sharpe) or hv_excess_sharpe <= 0):
    decision = "feature_only_or_veto_research"
else:
    decision = "discard_until_fixed"

discard_decision_report = pd.DataFrame([{
    "decision": decision,
    "standalone_alpha_promotable": decision == "promote_candidate",
    "primary_reason": "verified but weak IC does not convert into positive active portfolio performance after costs" if decision == "feature_only_or_veto_research" else "see failure_stage_report",
    "primary_benchmark": "equal_weight_active_universe_after_costs",
    "high_vol_5d_ic": hv_5d_ic,
    "high_vol_20bps_excess_sharpe": hv_excess_sharpe,
    "expanded_5d_ic": expanded_5d_ic,
    "expanded_20bps_excess_sharpe": expanded_excess_sharpe,
    "formula_checks_passed": formula_pass,
    "metric_checks_passed": metric_pass,
    "causal_orientation_passed": causal_pass,
}])
discard_decision_report.to_csv(VERIFICATION_ARTIFACTS["decision"], index=False)

display(failure_stage_report)
display(discard_decision_report)


## 25. Alpha#1 Salvage Experiments

This section tests the improvement ideas from the six-agent review without redownloading data and without synthetic inputs. The code is deliberately cache-first: after the first run, the notebook loads the generated CSV artifacts unless `RUN_SALVAGE_REFRESH = True`.

The experiments focus on the things most likely to rescue weak IC: causal 3d/5d orientation, signal smoothing, benchmark-relative overlays, partial rebalancing, no-trade bands, interaction features, residualized IC, and walk-forward regime-gate selection.

In [ ]:
from pathlib import Path

from research.alpha001_salvage_experiments import run_salvage_experiments

# Fast path: loads existing artifacts. Set True to recompute from cached OHLCV/mask CSVs.
RUN_SALVAGE_REFRESH = False

salvage_outputs = run_salvage_experiments(Path('.'), refresh=RUN_SALVAGE_REFRESH)

salvage_decision = salvage_outputs['decision']
salvage_grid = salvage_outputs['experiment_grid']
salvage_interactions = salvage_outputs['interaction']
salvage_neutralized_ic = salvage_outputs['neutralized_ic']
salvage_turnover = salvage_outputs['turnover_control']
salvage_oos_gates = salvage_outputs['oos_gate']

print('Salvage decision')
display(salvage_decision)

print('Best full-sample active overlays at 20 bps')
display(
    salvage_grid
    .query('cost_bps == 20.0')
    .sort_values(['active_sharpe', 'active_cagr'], ascending=False)
    [[
        'panel', 'strategy', 'cost_bps', 'active_sharpe', 'active_cagr',
        'alpha_cagr', 'benchmark_cagr', 'alpha_avg_daily_turnover', 'cost_drag_ann'
    ]]
    .head(15)
)

print('Best turnover-controlled variants at 20 bps')
display(
    salvage_turnover
    [[
        'panel', 'strategy', 'active_sharpe', 'active_cagr',
        'alpha_avg_daily_turnover', 'benchmark_avg_daily_turnover', 'turnover_excess'
    ]]
    .head(15)
)

print('Top interaction diagnostics by IC')
display(
    salvage_interactions
    .query('observations >= 500')
    .sort_values(['mean_rank_ic', 'rank_icir'], ascending=False)
    .head(20)
)

print('Residualized IC check')
display(
    salvage_neutralized_ic
    .sort_values(['panel', 'horizon_days', 'mean_rank_ic'], ascending=[True, True, False])
)

print('Nested walk-forward gate selection')
display(salvage_oos_gates)


## 26. Candidate Stress Test v1

This section freezes the most promising expression from the salvage layer: a causal 5-day Alpha#1 signal, EWM-3 smoothing, a 20% benchmark-relative overlay, weekly rebalancing, and 50% partial adjustment. It then asks whether the candidate survives stricter masks, ex-microcap checks, liquidity filters, capacity diagnostics, neutralized construction, and fold-level attribution.

This is still not a production approval. The main unresolved blocker is point-in-time constituent history for the expanded parent universe.

In [ ]:
from pathlib import Path

from research.alpha001_salvage_experiments import run_candidate_stress_tests

# Fast path: loads existing artifacts. Set True to recompute from cached OHLCV/mask CSVs.
RUN_CANDIDATE_STRESS_REFRESH = False

candidate_outputs = run_candidate_stress_tests(Path('.'), refresh=RUN_CANDIDATE_STRESS_REFRESH)

candidate_stress_decision = candidate_outputs['decision']
candidate_stress = candidate_outputs['stress']
candidate_folds = candidate_outputs['fold']
candidate_liquidity = candidate_outputs['liquidity']
candidate_exposure = candidate_outputs['exposure']
candidate_neutralized = candidate_outputs['neutralized_portfolio']
candidate_latest_portfolio = candidate_outputs['latest_portfolio']
candidate_gate_policy = candidate_outputs['gate_policy']
candidate_pit_gap = candidate_outputs['pit_gap']

print('Candidate stress decision')
display(candidate_stress_decision)

print('Gate policy comparison')
display(candidate_gate_policy)

print('Point-in-time data gap')
display(candidate_pit_gap)

print('Stress variants at 20 bps')
display(
    candidate_stress
    .query('cost_bps == 20.0')
    .sort_values(['panel', 'active_sharpe'], ascending=[True, False])
    [[
        'panel', 'mask', 'active_sharpe', 'active_cagr', 'alpha_cagr', 'benchmark_cagr',
        'alpha_avg_daily_turnover', 'median_active_names', 'p10_active_names'
    ]]
)

print('Fold-level active performance at 20 bps')
display(
    candidate_folds
    .query('cost_bps == 20.0')
    .sort_values(['panel', 'mask', 'fold_start'])
)

print('Neutralized candidate portfolio check')
display(
    candidate_neutralized
    .query('cost_bps == 20.0')
    [[
        'panel', 'strategy', 'active_sharpe', 'active_cagr', 'alpha_cagr',
        'benchmark_cagr', 'alpha_avg_daily_turnover'
    ]]
    .sort_values(['panel', 'active_sharpe'], ascending=[True, False])
)

print('Latest candidate portfolio snapshots')
display(
    candidate_latest_portfolio
    .query("mask in ['base', 'ex_microcap_strict_liquidity_100m', 'seasoned_2y_strict_liquidity_100m']")
    .sort_values(['panel', 'mask', 'active_weight'], ascending=[True, True, False])
    [[
        'date', 'panel', 'mask', 'symbol', 'candidate_weight', 'benchmark_weight', 'active_weight',
        'alpha001_signal', 'is_current_benchmark_member', 'is_carried_position',
        'lagged_60d_median_adv_rupees', 'industry', 'source_slugs'
    ]]
    .head(60)
)

print('Liquidity and capacity diagnostics')
display(
    candidate_liquidity
    .query("mask in ['base', 'ex_microcap_strict_liquidity_100m', 'seasoned_2y_strict_liquidity_100m']")
    .sort_values(['panel', 'mask', 'portfolio', 'metric'])
)

print('Expanded universe exposure diagnostics')
display(
    candidate_exposure
    .query("panel == 'expanded_high_vol_top100' and mask in ['base', 'ex_microcap', 'ex_microcap_strict_liquidity_100m', 'seasoned_2y_strict_liquidity_100m']")
    .sort_values(['mask', 'portfolio', 'exposure_type'])
)


## Promotion Checklist

Use the notebook outputs in this order:

1. Data checkpoint must show enough usable symbols and no structural OHLC/price failures.
2. Oriented IC should be positive at the chosen horizon and not only from one short period.
3. Bucket spread should be positive after orientation.
4. Simple basket metrics should survive costs.
5. Full-sample rule search is allowed to discover ideas, but promotion depends on walk-forward OOS metrics.
6. Candidate table is an alpha output, not a trade ticket; review liquidity, risk, sector exposure, and execution before use.